# Imports & Parameters

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'


## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [2]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)

data_dumper_path = '/Users/rapido/local-datasets/customer/appography/appography_data_dump/'
local_database = '/Users/rapido/local-datasets/customer/appography/appography_v2/bangalore/database/'

/Users/rapido/analytics/customer/Appography/appography_demographic_dataset


In [3]:
##Parameter 
start_date = '20240301'
end_date = '20240331'
city = 'Bangalore'

city_service = f"""
    select 
        service_level city_service
    from 
        datasets.service_mapping 
    where 
        city_display_name = '{city}' 
        and service_type != 'delivery'  
        and service_category != 'delivery'
"""
df_city_service = pd.read_sql(city_service, connection)
service = ', '.join("'" + value + "'" for value in sorted(df_city_service['city_service']))

In [4]:
print("\033[1m Date Range : \033[0m", start_date, " to ", end_date)
print("\033[1m City : \033[0m", city)
print("\033[1m Service : \033[0m", service)

 Date Range :  20240301  to  20240331
 City :  Bangalore
 Service :  'Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CityAuto', 'Link', 'Lite'


# Datasets

In [5]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)

/Users/rapido/analytics/customer/Appography/appography_demographic_dataset


## Customer base

### Metric data

print('Query 1 running..')
start_time = time.time()

active_customer_base = f"""

    WITH fe_tbl AS (
    
        SELECT 
            yyyymmdd,
            platform,
            service_detail_id,
            user_id customer_id,
            fare_estimate_id,
            quarter_hour fe_quarter_hour
        FROM 
            pricing.fare_estimates_enriched 
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city = '{city}'
            AND service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CityAuto', 'Link', 'Lite')
            AND platform IN ('android', 'ios')
        ),
        
        rr_net_tbl as (
        
        SELECT   
            yyyymmdd,
            service_detail_id,
            customer_id,
            estimate_id AS fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag,
            quarter_hour rr_quarter_hour
        FROM 
            orders.order_logs_snapshot ols
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city_name = '{city}'
            AND service_obj_service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CityAuto', 'Link', 'Lite')
        ),
        
        merged AS (
        
        SELECT
            fe_tbl.*,
            rr_net_tbl.order_id,
            rr_net_tbl.order_status,
            rr_net_tbl.spd_fraud_flag,
            rr_net_tbl.rr_quarter_hour
        FROM
            fe_tbl
        LEFT JOIN
            rr_net_tbl
            ON fe_tbl.yyyymmdd = rr_net_tbl.yyyymmdd
            AND fe_tbl.customer_id = rr_net_tbl.customer_id
            AND fe_tbl.service_detail_id = rr_net_tbl.service_detail_id
        )

        SELECT
            platform,
            customer_id,
            COUNT(DISTINCT fare_estimate_id) AS fe_count,
            COUNT(DISTINCT order_id) AS rr_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count,

            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(fe_quarter_hour,1,2) AS REAL) BETWEEN 8 AND 12 THEN fare_estimate_id END) morning_fe,
            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(fe_quarter_hour,1,2) AS REAL) BETWEEN 13 AND 22 THEN fare_estimate_id END) evening_fe,

            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(rr_quarter_hour,1,2) AS REAL) BETWEEN 8 AND 12 THEN order_id END) morning_rr,
            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(rr_quarter_hour,1,2) AS REAL) BETWEEN 13 AND 22 THEN order_id END) evening_rr

        FROM 
            merged
        GROUP BY 1,2
        
"""
df_active_customer_base = pd.read_sql(active_customer_base, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_active_customer_base.head(1)

In [6]:
# df_active_customer_base.to_csv(local_database + 'raw_active_customer_base.csv', index=False)
df_active_customer_base = pd.read_csv(local_database + 'raw_active_customer_base.csv')

### Segment data

print('Query 2 running..')
start_time = time.time()
customer_segments = f"""

     SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_lifetime_rides life_time_rides,
        taxi_lifetime_stage life_time_stage,
        taxi_service_affinity service_affinity

    FROM 
        datasets.iallocator_customer_segments
    WHERE 
        run_date =  DATE_FORMAT(DATE_PARSE('{end_date}', '%Y%m%d') + INTERVAL '1' DAY , '%Y-%m-%d')
        AND taxi_recency_segment != 'INACTIVE' 
        
"""
df_customer_segments = pd.read_sql(customer_segments, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_segments.head(1)

In [7]:
# df_customer_segments.to_csv(local_database + 'raw_customer_segments.csv', index=False)
df_customer_segments = pd.read_csv(local_database + 'raw_customer_segments.csv')

### Age and gender data

print('Query 3 running..')
start_time = time.time()

customer_demo_data = f"""

        WITH fe_customer AS (

        SELECT 
            user_id customer_id
        FROM 
            pricing.fare_estimates_enriched 
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city = '{city}'
            AND service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CityAuto', 'Link', 'Lite')
            AND platform IN ('android', 'ios')
        GROUP BY 1
        ),

        demographic_data AS ( 

        SELECT
            customerId customer_id,
            CASE WHEN age BETWEEN 15 AND 60 THEN age ELSE 0 END age,
            COALESCE(gender, 'UNKNOWN') gender
        FROM 
            datasets.customer_single_view
        )

        SELECT
            a.customer_id,
            COALESCE(age, 0) age,
            COALESCE(gender, 'UNKNOWN') gender
        FROM 
            fe_customer a
        LEFT JOIN 
            demographic_data b
            on a.customer_id = b.customer_id        
"""

df_customer_demo_data = pd.read_sql(customer_demo_data, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_demo_data.head(1)

In [8]:
# df_customer_demo_data.to_csv(local_database + 'raw_customer_demo_data.csv', index=False)
df_customer_demo_data = pd.read_csv(local_database + 'raw_customer_demo_data.csv')

### Use-case data

print('Query 4 running..')
start_time = time.time()

customer_use_case_data = f"""

    WITH use_case AS (

        SELECT
            hex_10, 
            usecase_tag
        FROM
            (
            SELECT 
                hex_10, 
                combined_final_usecase_accuracy as usecase_tag,
                ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
            FROM
                hive.experiments_internal.combined_usecase_hex_10_level
            WHERE 
                aoi = 'Bangalore, India'
            )
        WHERE   
            seq_no = 1
        ),

        active_customers AS (

        SELECT 
            customer_id,
            order_id,
            drop_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'first_half_drop' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON drop_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 8 
            AND cast(substr(quarter_hour,1,2) as real) <= 12    
            AND city_name = '{city}'
            
        GROUP BY 1,2,3,4

        UNION 

            SELECT 
            customer_id,
            order_id,
            pickup_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'second_half_pickup' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON pickup_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 13
            AND cast(substr(quarter_hour,1,2) as real) <= 21
            AND city_name = '{city}'
        GROUP BY 1,2,3,4

        ),

        merge AS (

        SELECT
            customer_id,
            time_period,
            usecase_tag,
            orders,
            ROW_NUMBER() OVER(PARTITION BY customer_id,time_period ORDER BY orders DESC) seq_no
        FROM 
            (
            SELECT
                customer_id,
                time_period,
                usecase_tag,
                COUNT(DISTINCT order_id) orders
            FROM 
                active_customers
            GROUP BY 1,2,3
            )
        -- WHERE 
        --     LOWER(usecase_tag) != 'unknown'
        )

        SELECT
            customer_id,
            COALESCE(first_half_drop_primary, 'NA') first_half_drop_primary,
            COALESCE(first_half_drop_secondary, 'NA') first_half_drop_secondary,
            COALESCE(second_half_pickup_primary, 'NA') second_half_pickup_primary,
            COALESCE(second_half_pickup_secondary, 'NA') second_half_pickup_secondary
        FROM
        (
        SELECT
            customer_id,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 1 THEN usecase_tag END) first_half_drop_primary,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 2 THEN usecase_tag END) first_half_drop_secondary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 1 THEN usecase_tag END) second_half_pickup_primary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 2 THEN usecase_tag END) second_half_pickup_secondary
        FROM 
            merge 
        WHERE 
            seq_no in (1,2)
        GROUP BY 1
        ) 
"""

df_customer_use_case_data = pd.read_sql(customer_use_case_data, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_use_case_data.head(1)

In [9]:
# df_customer_use_case_data.to_csv(local_database + 'raw_customer_use_case_data.csv', index=False)
df_customer_use_case_data = pd.read_csv(local_database + 'raw_customer_use_case_data.csv')

### Appography data

print('Query 5 running..')
start_time = time.time()

date_range = pd.date_range(start=start_date, end=end_date, freq='D').strftime('%Y%m%d')
file_names = [f"users_app_list_{date}.csv" for date in date_range]
csv_files = [file for file in os.listdir(data_dumper_path) if file.endswith('.csv') and file in file_names]
#csv_files = ['users_app_list_20240325.csv'] # testing...

dfs = []

#Loop through each CSV file, read it into a DataFrame, and append to the list
for file in csv_files:
    file_path = os.path.join(data_dumper_path, file)
    df = pd.read_csv(file_path)
    dfs.append(df)

#Concatenate all DataFrames in the list into a single DataFrame
combined_data_dumper = pd.concat(dfs, ignore_index=True)

print("Fetched and Concatenated all files")

#Only selected city
df_combined_data_dumper = pd.merge(df_customer_demo_data['customer_id'],
                                   combined_data_dumper,
                                   how='inner',
                                   left_on=['customer_id'],
                                   right_on=['user_id'] 
                                  ) 
print("Merged")

#delete combined_data_dumper
del combined_data_dumper

#Latest updated data
df_combined_data_dumper['last_sync_date'] = df_combined_data_dumper['last_sync_date'].astype('int32')
df_combined_data_dumper = df_combined_data_dumper.sort_values(by=['user_id', 'last_sync_date'], ascending=[True, False])
df_combined_data_dumper = df_combined_data_dumper.drop_duplicates(subset='user_id', keep='first')
df_combined_data_dumper = df_combined_data_dumper[['customer_id', 'app_list_set']]

print("Updated data")

#regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_combined_data_dumper['app_list_set'] = df_combined_data_dumper['app_list_set'].apply(lambda x: extract_and_trim(x))
df_customer_appography_data = df_combined_data_dumper[['customer_id', 'app_list_set']]

print("regular-exp")

#delete df_combined_data_dumper
del df_combined_data_dumper

end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time : ", execution_time, " mins")

In [10]:
# df_customer_appography_data.to_csv(local_database + 'raw_customer_appography_data.csv', index=False)
df_customer_appography_data = pd.read_csv(local_database + 'raw_customer_appography_data.csv')

# regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_customer_appography_data['app_list_set'] = df_customer_appography_data['app_list_set'].apply(lambda x: extract_and_trim(x))

### Merge

In [11]:
df_active_customer_base.head(1)

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr
0,android,63ff28e9f22ead6bac895265,20,12,8,8,11,1,10


In [12]:
df_customer_segments.head(1)

,customer_id,ltr_segment,life_time_rides,life_time_stage,service_affinity
0,6064a0c0193b5c16f90563d8,UNKNOWN,0,UNKNOWN,UNKNOWN


In [13]:
df_customer_demo_data.head(1)

,customer_id,age,gender
0,64b7c46ab5f93bd98388dec9,0,MALE


In [14]:
df_customer_use_case_data.head(1)

,customer_id,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,63bd2c0a4fd273663b4be401,office,place_of_worship,NaN,NaN


In [15]:
df_customer_appography_data.head(1)

,customer_id,app_list_set
0,5737c6adddbec2203f73316d,"[whatsapp, paytm money, samsung galaxy wearabl..."


In [16]:
start_time = time.time()
print('Merging 1.. ')
merge1 = pd.merge(df_active_customer_base,df_customer_segments,how='left',on = ['customer_id'])

print('Merging 2..')
merge2 = pd.merge(merge1,df_customer_demo_data,how='left',on = ['customer_id'])

print('Merging 3..')
merge3 = pd.merge(merge2,df_customer_use_case_data,how='left',on = ['customer_id'])

print('Merging 4..')
merge4 = pd.merge(merge3,df_customer_appography_data,how='left',on = ['customer_id'])

df_all_customer = merge4

del merge1
del merge2
del merge3
del merge4

end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_all_customer.head(1)

Merging 1.. 
Merging 2..
Merging 3..
Merging 4..
Execution time: 0.98 mins


,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
0,android,63ff28e9f22ead6bac895265,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,0,MALE,leisure,NaN,residential,NaN,"[bookmyshow, youtube, jio cinema, google maps,..."


In [17]:
# df_all_customer.to_csv(local_database + 'processed_all_customer.csv', index=False)
# df_all_customer = pd.read_csv(local_database + 'processed_all_customer.csv')

# regular-exp 
# def extract_and_trim(text):
#     return [val.strip() for val in re.findall(r"'([^']*)'", text)]

# df_all_customer['app_list_set'] = df_all_customer['app_list_set'].apply(lambda x: extract_and_trim(x))

In [18]:
df_all_customer.head(1)

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
0,android,63ff28e9f22ead6bac895265,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,0,MALE,leisure,NaN,residential,NaN,"[bookmyshow, youtube, jio cinema, google maps,..."


In [19]:
df_all_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2766893 entries, 0 to 2766892
Data columns (total 20 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   platform                      object 
 1   customer_id                   object 
 2   fe_count                      int64  
 3   rr_count                      int64  
 4   net_count                     int64  
 5   morning_fe                    int64  
 6   evening_fe                    int64  
 7   morning_rr                    int64  
 8   evening_rr                    int64  
 9   ltr_segment                   object 
 10  life_time_rides               float64
 11  life_time_stage               object 
 12  service_affinity              object 
 13  age                           int64  
 14  gender                        object 
 15  first_half_drop_primary       object 
 16  first_half_drop_secondary     object 
 17  second_half_pickup_primary    object 
 18  second_half_pickup_sec

In [20]:
df_all_customer['fe_count'] = df_all_customer['fe_count'].astype('int32')
df_all_customer['rr_count'] = df_all_customer['rr_count'].astype('int32')
df_all_customer['net_count'] = df_all_customer['net_count'].astype('int32')
df_all_customer['morning_fe'] = df_all_customer['morning_fe'].astype('int32')
df_all_customer['evening_fe'] = df_all_customer['evening_fe'].astype('int32')
df_all_customer['morning_rr'] = df_all_customer['morning_rr'].astype('int32')
df_all_customer['evening_rr'] = df_all_customer['evening_rr'].astype('int32')
df_all_customer['age'] = df_all_customer['age'].astype('int32')

df_all_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2766893 entries, 0 to 2766892
Data columns (total 20 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   platform                      object 
 1   customer_id                   object 
 2   fe_count                      int32  
 3   rr_count                      int32  
 4   net_count                     int32  
 5   morning_fe                    int32  
 6   evening_fe                    int32  
 7   morning_rr                    int32  
 8   evening_rr                    int32  
 9   ltr_segment                   object 
 10  life_time_rides               float64
 11  life_time_stage               object 
 12  service_affinity              object 
 13  age                           int32  
 14  gender                        object 
 15  first_half_drop_primary       object 
 16  first_half_drop_secondary     object 
 17  second_half_pickup_primary    object 
 18  second_half_pickup_sec

## Distribution check
### Platform customer distribution

In [21]:
## platform customer distribution
df_temp = df_all_customer.groupby(['platform']).agg(customers = ('customer_id', 'nunique')).reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,customers,% distribution
0,android,2270829,82.07
1,ios,496064,17.93


### Age customer distribution

In [22]:
df_all_customer['age_presence'] = np.where(df_all_customer['age'] > 0, 'AVAILABLE', 'NA')

df_temp = df_all_customer\
                .groupby(['age_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,age_presence,customers,% distribution
0,AVAILABLE,955338,34.61
1,NA,1805166,65.39


In [23]:
df_temp = df_all_customer\
                .groupby(['platform', 'age_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['total'] = df_temp.groupby('platform')['customers'].transform('sum')
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp['total'], 2)
df_temp

,platform,age_presence,customers,total,% distribution
0,android,AVAILABLE,755034,2270829,33.25
1,android,NA,1515795,2270829,66.75
2,ios,AVAILABLE,203139,496064,40.95
3,ios,NA,292925,496064,59.05


### Gender distribution

In [24]:
df_temp = df_all_customer\
                .groupby(['gender'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,gender,customers,% distribution
0,FEMALE,753286,27.29
1,MALE,1926225,69.78
2,OTHERS,8657,0.31
3,UNKNOWN,72336,2.62


In [25]:
df_temp = df_all_customer\
                .groupby(['platform', 'gender'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['total'] = df_temp.groupby('platform')['customers'].transform('sum')
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp['total'], 2)
df_temp

,platform,gender,customers,total,% distribution
0,android,FEMALE,592616,2270829,26.10
1,android,MALE,1599463,2270829,70.44
2,android,OTHERS,7745,2270829,0.34
3,android,UNKNOWN,71005,2270829,3.13
4,ios,FEMALE,162571,496064,32.77
5,ios,MALE,331234,496064,66.77
6,ios,OTHERS,927,496064,0.19
7,ios,UNKNOWN,1332,496064,0.27


### Appography distribution

In [26]:
df_all_customer['appo_presence'] = np.where(df_all_customer['app_list_set'].isna() == True, 'NA', 'AVAILABLE')

df_temp = df_all_customer[df_all_customer['platform'] == 'android']\
                .groupby(['platform', 'appo_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,appo_presence,customers,% distribution
0,android,AVAILABLE,1877684,82.69
1,android,NA,393145,17.31


### Opportunity Size

In [27]:
df_all_customer['appo_presence'] = np.where(df_all_customer['app_list_set'].isna() == True, 'NA', 'AVAILABLE')

df_temp = df_all_customer[df_all_customer['platform'] == 'android']\
                .groupby(['platform', 'age_presence', 'appo_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,age_presence,appo_presence,customers,% distribution
0,android,AVAILABLE,AVAILABLE,623111,27.44
1,android,AVAILABLE,NA,131923,5.81
2,android,NA,AVAILABLE,1254573,55.25
3,android,NA,NA,261222,11.50


In [28]:
df_temp.pivot(index ='age_presence' , columns ='appo_presence', values =['% distribution'])

% distribution       
appo_presence      AVAILABLE     NA
age_presence                       
AVAILABLE              27.44   5.81
NA                     55.25  11.50

## Triangulation 1 - Appography Age Bins Classification

In [29]:
df_all_customer.columns

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 'ltr_segment',
       'life_time_rides', 'life_time_stage', 'service_affinity', 'age',
       'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
       'second_half_pickup_primary', 'second_half_pickup_secondary',
       'app_list_set', 'age_presence', 'appo_presence'],
      dtype='object')

In [30]:
df_triangulation_1 = df_all_customer[(df_all_customer['platform'].isin(['android']) 
                                      & 
                                      df_all_customer['appo_presence'].isin(['AVAILABLE']))]\
                                    [['customer_id', 'age', 'appo_presence','app_list_set']]

df_triangulation_1.head(5)

,customer_id,age,appo_presence,app_list_set
0,63ff28e9f22ead6bac895265,0,AVAILABLE,"[bookmyshow, youtube, jio cinema, google maps,..."
1,5cb7f15554bc7263ff3f4e73,28,AVAILABLE,"[google photos, pinterest, snapchat, youtube, ..."
2,62428c959d4b75f09cb3a511,0,AVAILABLE,"[zee5, meesho, youtube, flipkart, gmail, googl..."
4,5fbaaf067e483f4a13fd9b9e,28,AVAILABLE,"[whatsapp, swiggy, google maps, youtube, spoti..."
5,64d0d9ebc810cc925f36ceea,0,AVAILABLE,"[truecaller, google maps, youtube, flipkart, m..."


### Explode

In [31]:
## Explode list into individual rows

df_appography_explode = df_triangulation_1.explode('app_list_set')
df_appography_explode['app_list_set'] = df_appography_explode['app_list_set'].str.lower()

df_appography_explode.head(4)

,customer_id,age,appo_presence,app_list_set
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow
0,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube
0,63ff28e9f22ead6bac895265,0,AVAILABLE,jio cinema
0,63ff28e9f22ead6bac895265,0,AVAILABLE,google maps


### Mobile app to category mapping

In [32]:
category_list = pd.read_csv('/Users/rapido/local-datasets/customer/appography/appography_v1/category_list_v1.csv')
category_list['app_name'] = category_list['app_name'].str.lower()
category_list['categories_l1'] = category_list['categories_l1'].str.lower()
category_list['categories_l2'] = category_list['categories_l2'].str.lower()
category_list = category_list[['app_name', 'categories_l1', 'categories_l2']]
print(category_list.shape)
print(category_list.categories_l1.unique())
print(category_list.categories_l2.unique())
category_list.head(1)

(399, 3)
['educational' 'ott' 'delivery_grocery' 'tools' 'office' 'dating' 'news'
 'devotional' 'travel_metro' 'gaming' 'travel_bookings'
 'finance_transactions' 'ecommerce' 'streaming_music' 'finance_investment'
 'health' 'fantasy_sports' 'delivery_food' 'home_security' 'finance_news'
 'travel_ridehailing' 'social' 'telecom' 'entertainment' 'news_finance'
 'wearable' 'driver_app']
['educational' 'rest' 'office' 'ride haling' 'gaming' 'finance_investment'
 'driver_app']


,app_name,categories_l1,categories_l2
0,ignou e-content,educational,educational


In [33]:
## Derived category

core_education = ['aakash','chegg study','google classroom','microsoft math solver','simplilearn','moodle','vedantu','vidyakul','diksha','e-pg pathshala','mycbseguide','physics wallah']
core_office = ['slack','asana','confluence','jira','cisco','github','google analytics','miro','trello','zoho mail','zoho meeting']
top_categories = ['dating', 'delivery_food', 'driver_app', 'entertainment', 'finance_investment', 'gaming', 'office', 'social', 'streaming_music', 'travel_bookings', 'travel_ridehailing']


category_list['top_categories'] = np.where(category_list['app_name'].isin(core_education), 'core_education',  #category_list['app_name'],
                                    np.where(category_list['app_name'].isin(core_office), 'core_office', #category_list['app_name'],
                                        np.where(category_list['categories_l1'].isin(top_categories), category_list['categories_l1'], 'rest'
                                            )))

In [34]:
top_apps_list = category_list[category_list['top_categories'] != 'rest']['app_name'].values.tolist()

In [35]:
## Top apps
# df_main_apps_explode = df_appography_explode[df_appography_explode['app_list_set'].isin(top_apps_list)]

df_main_apps_explode = df_appography_explode

### Merge

In [36]:
df_main_apps_explode.head(2)

,customer_id,age,appo_presence,app_list_set
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow
0,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube


In [37]:
category_list.head(2)

,app_name,categories_l1,categories_l2,top_categories
0,ignou e-content,educational,educational,rest
1,aha,ott,rest,rest


In [38]:
df_main_app_category =  pd.merge(df_main_apps_explode, 
                                 category_list, 
                                 how='left', 
                                 left_on='app_list_set', 
                                 right_on='app_name'
                              )
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_list_set,app_name,categories_l1,categories_l2,top_categories
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow,bookmyshow,entertainment,rest,entertainment
1,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube,youtube,tools,rest,rest
2,63ff28e9f22ead6bac895265,0,AVAILABLE,jio cinema,jio cinema,ott,rest,rest
3,63ff28e9f22ead6bac895265,0,AVAILABLE,google maps,google maps,tools,rest,rest


#### Distribution check on categories

In [39]:
total_customers = df_main_app_category.customer_id.nunique()
total_customers

df_temp = df_main_app_category\
            .groupby(['categories_l1'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,customers,%
0,tools,"[youtube, google maps, notion, clock, adobe li...",1877625,100.00
1,social,"[telegram, skype, line, instagram, twitter, vi...",1875178,99.87
2,ott,"[sonyliv, voot, jio cinema, amazon prime video...",1527173,81.33
3,ecommerce,"[myntra, club factory, nykaa, zivame, shiprock...",1501596,79.97
4,travel_ridehailing,"[blusmart, quick ride, jugnoo, in drive, drive...",1335369,71.12
5,finance_transactions,"[cred, mobikwik, freecharge, axis mobile, payt...",1264156,67.33
6,streaming_music,"[hungama music, amazon music, gaana, wynk musi...",1254092,66.79
7,delivery_food,"[zomato, dunzo, uber eats, swiggy]",1153487,61.43
8,office,"[google authenticator, confluence, onedrive, a...",1067225,56.84
9,finance_investment,"[groww, namma partner, hdfc mobile banking, ki...",662382,35.28


#### Most Common used apps

In [40]:
df_temp = df_main_app_category\
            .groupby(['app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,app_list_set,customers,%
0,google maps,1877029,99.97
1,gmail,1873112,99.76
2,youtube,1873057,99.75
3,whatsapp,1830156,97.47
4,google photos,1775553,94.56
5,instagram,1523428,81.13
6,facebook,1310924,69.82
7,flipkart,1273245,67.81
8,truecaller,1221064,65.03
9,google calendar,1187551,63.25


In [46]:
most_common_apps = ['google maps', 'gmail', 'youtube', 'whatsapp', 'clock', 'calculator',
                    'google calendar', 'google photos']

### Way 1 (All Apps Level)

#### Transform Categories 

In [47]:
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_list_set,app_name,categories_l1,categories_l2,top_categories
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow,bookmyshow,entertainment,rest,entertainment
1,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube,youtube,tools,rest,rest
2,63ff28e9f22ead6bac895265,0,AVAILABLE,jio cinema,jio cinema,ott,rest,rest
3,63ff28e9f22ead6bac895265,0,AVAILABLE,google maps,google maps,tools,rest,rest


In [48]:
df_customer_mapped_v1 = df_main_app_category[~df_main_app_category['app_list_set'].isin(most_common_apps)]\
                        .groupby(['customer_id', 'age'])\
                        .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))),
                             top_categories = ('top_categories', lambda x: list(set(x))))\
                        .reset_index()

df_customer_mapped_v1.head(4)

,customer_id,age,app_list,categories_l1,top_categories
0,5737c6adddbec2203f73316d,0,"[onedrive, disney+ hotstar, cred, rapido capta...","[social, wearable, ott, driver_app, tools, tra...","[rest, social, driver_app, travel_ridehailing,..."
1,5737c6aeddbec2203f733176,34,"[cred, groww, instagram, zomato, onedrive, mic...","[finance_investment, delivery_grocery, social,...","[rest, finance_investment, social, delivery_fo..."
2,5737c6b0ddbec2203f733182,28,"[zee5, ola driver app, disney+ hotstar, uber, ...","[entertainment, social, ott, delivery_food, to...","[rest, entertainment, social, delivery_food, t..."
3,5737c6b1ddbec2203f73318b,33,"[twitter, zee5, disney+ hotstar, uber, rapido ...","[travel_bookings, news, delivery_grocery, soci...","[rest, travel_bookings, social, delivery_food,..."


In [49]:
unique_categories = set(category for sublist in df_customer_mapped_v1['app_list'] for category in sublist)
unique_categories_list = list(unique_categories)
print("Count of unique apps:", len(unique_categories))
sorted(unique_categories)

Count of unique apps: 212


['aakash',
 'aastha tv',
 'accuweather',
 'adobe lightroom',
 'adobe scan',
 'aha',
 'airbnb',
 'aisle',
 'ajio',
 'alt balaji',
 'amazon music',
 'amazon prime video',
 'amazon shopping',
 'angel broking',
 'asana',
 'axis mobile',
 'bansal classes',
 'bigbasket',
 'blusmart',
 'booking.com',
 'bookmyshow',
 'borzo driver app',
 'brainly',
 'bumble',
 'byju',
 'caclubindia',
 'chegg study',
 'cisco',
 'cleartrip',
 'club factory',
 'coffee meets bagel',
 'colegeduniya',
 'collegedekho',
 'confluence',
 'course hero',
 'coursera',
 'cred',
 'cricplay',
 'diksha',
 'disney+ hotstar',
 'doubtnut',
 'driveu driver app',
 'dropbox',
 'dunzo',
 'dunzo driver app',
 'duolingo',
 'e-pg pathshala',
 'economic times',
 'edx',
 'embibe',
 'facebook',
 'feedly',
 'fiitjee',
 'file commander',
 'firstcry',
 'fitbit',
 'fiton',
 'flipboard',
 'flipkart',
 'fossil smartwatches',
 'freecharge',
 'gaana',
 'garmin connect',
 'geeks for geeks',
 'github',
 'goibibo',
 'goodreads',
 'google analytics',


In [50]:
for category in unique_categories:
    df_customer_mapped_v1[category] = df_customer_mapped_v1['app_list'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v1.head(1)

,customer_id,age,app_list,categories_l1,top_categories,myfitnesspal,onedrive,aisle,the indian express,aha,freecharge,garmin connect,feedly,hungama music,soundcloud,cred,mlb,microsoft lens,sonyliv,coursera,colegeduniya,zomato,zoom,google classroom,vidyakul,...,redbus,amazon music,google sheets,uber eats,samsung galaxy wearable,dunzo,paytm money,chegg study,zomato driver app,vedantu,grofers,instagram,goibibo,ignou e-content,udemy,zoho mail,mx takatak,kite by zerodha,roposo,nba,driveu driver app,namma yatri,edx,stack overflow,indigo
0,5737c6adddbec2203f73316d,0,"[onedrive, disney+ hotstar, cred, rapido capta...","[social, wearable, ott, driver_app, tools, tra...","[rest, social, driver_app, travel_ridehailing,...",0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,...,0,0,1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [51]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['15-20', '21-25',  '26-30', '31-35', '36-45', '46-60']

# bins = [15, 20, 25, 35, 61]
# labels = ['15-20', '21-25',  '26-35', '36-60']

df_customer_mapped_v1['age_group'] = pd.cut(df_customer_mapped_v1['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [52]:
main_list = ['customer_id', 'age', 'age_group' ,'categories_l1']
unique_column_list = main_list + unique_categories_list

In [53]:
df_customer_mapped_v1 = df_customer_mapped_v1[unique_column_list]
df_customer_mapped_v1.head(1)

,customer_id,age,age_group,categories_l1,myfitnesspal,onedrive,aisle,the indian express,aha,freecharge,garmin connect,feedly,hungama music,soundcloud,cred,mlb,microsoft lens,sonyliv,coursera,colegeduniya,zomato,zoom,google classroom,vidyakul,slack,...,redbus,amazon music,google sheets,uber eats,samsung galaxy wearable,dunzo,paytm money,chegg study,zomato driver app,vedantu,grofers,instagram,goibibo,ignou e-content,udemy,zoho mail,mx takatak,kite by zerodha,roposo,nba,driveu driver app,namma yatri,edx,stack overflow,indigo
0,5737c6adddbec2203f73316d,0,NaN,"[social, wearable, ott, driver_app, tools, tra...",0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [54]:
df_customer_mapped_v1_age_w = df_customer_mapped_v1[~df_customer_mapped_v1['age_group'].isna()]
df_customer_mapped_v1_age_na = df_customer_mapped_v1[df_customer_mapped_v1['age_group'].isna()]
df_customer_mapped_v1_age_w.drop('categories_l1', axis=1, inplace=True)
df_customer_mapped_v1_age_na.drop('categories_l1', axis=1, inplace=True)

#### Model 

In [55]:
# Step 1: Training the Model
X = df_customer_mapped_v1_age_w.drop(columns=['customer_id', 'age', 'age_group'])
y = df_customer_mapped_v1_age_w['age_group']

# Encode categorical variables if needed
label_encoder = LabelEncoder()
for col in X.columns:
    X[col] = label_encoder.fit_transform(X[col])
    
    
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)


model = RandomForestClassifier()  # Random Forest Classifier
model.fit(X_train, y_train)

RandomForestClassifier()

In [56]:
# Use the trained model to make predictions on the test set
y_pred = model.predict(X_test)

# Get unique target values
unique_classes = np.unique(y_test)

# Initialize dictionaries to store metrics for each target value
accuracy_dict = {}
precision_dict = {}
recall_dict = {}
f1_score_dict = {}

from sklearn.metrics import f1_score

# Calculate metrics for each target value
for class_name in unique_classes:
    # Filter predictions and true labels for the current target value
    y_true_class = (y_test == class_name)
    y_pred_class = (y_pred == class_name)

    # Calculate accuracy, recall, F1-score
    accuracy_dict[class_name] = accuracy_score(y_true_class, y_pred_class)
    precision_dict[class_name] = precision_score(y_true_class, y_pred_class)
    recall_dict[class_name] = recall_score(y_true_class, y_pred_class)
    f1_score_dict[class_name] = f1_score(y_true_class, y_pred_class)

In [57]:
for class_name in unique_classes:
    print(f"Target Value: {class_name}")
    print("Accuracy:", round(accuracy_dict[class_name] * 100.00, 2), "%")
    print("Precision:", round(precision_dict[class_name] * 100.00, 2), "%")
    print("Recall:", round(recall_dict[class_name] * 100.00, 2), "%")
    print("F1 Score:", round(f1_score_dict[class_name] * 100.00, 2), "%")
    print()

Target Value: 15-20
Accuracy: 95.3 %
Precision: 23.64 %
Recall: 2.5 %
F1 Score: 4.53 %

Target Value: 21-25
Accuracy: 75.44 %
Precision: 47.91 %
Recall: 47.54 %
F1 Score: 47.72 %

Target Value: 26-30
Accuracy: 56.89 %
Precision: 40.52 %
Recall: 65.71 %
F1 Score: 50.13 %

Target Value: 31-35
Accuracy: 75.1 %
Precision: 33.25 %
Recall: 21.89 %
F1 Score: 26.4 %

Target Value: 36-45
Accuracy: 82.7 %
Precision: 35.81 %
Recall: 21.46 %
F1 Score: 26.84 %

Target Value: 46-60
Accuracy: 95.98 %
Precision: 20.51 %
Recall: 1.98 %
F1 Score: 3.61 %



In [58]:
# Calculate the classification report
report = classification_report(y_test, y_pred, output_dict=True)

# Extract accuracy, precision, recall, and F1-score for each target value
overall_accuracy = report['accuracy']
overall_precision = report['macro avg']['precision']
overall_recall = report['macro avg']['recall']
overall_f1_score = report['macro avg']['f1-score']

print("Model Accuracy:", round(overall_accuracy*100.00, 2), "%")
print("Precision:", round(overall_precision*100.00, 2), "%")
print("Recall:", round(overall_recall*100.00, 2), "%")
print("F1 Score:", round(overall_f1_score*100.00, 2), "%")

Model Accuracy: 40.7 %
Precision: 33.61 %
Recall: 26.85 %
F1 Score: 26.54 %


#### Feature selection 

In [62]:
# Get feature importances
feature_importances = model.feature_importances_

# Map feature names to their importance scores
feature_importance_dict = dict(zip(X.columns, feature_importances))

# Iterate through each target value
for class_name in unique_classes:
    print(f"Target Value: {class_name}")
    
    # Filter the data for the current target value
    X_test_class = X_test[y_test == class_name]
    y_pred_class = y_pred[y_test == class_name]
    
    # Check if there are any instances of the current target value in the test set
    if len(X_test_class) > 0:
        # Get feature importances for the current target value
        feature_importance_class = {feature: importance for feature, importance in feature_importance_dict.items()}
        
        # Sort feature importances by importance score
        sorted_features = sorted(feature_importance_class.items(), key=lambda x: x[1], reverse=True)
        
        # Limit the number of top features displayed
        top_features = sorted_features[:10]  # Change the number here to display more or fewer top features
        
        # Print top features
        print("Top Features:")
        for feature, importance in top_features:
            print(f"{feature}: {importance}")
        print()
    else:
        print("No instances of this target value in the test set.")

Target Value: 15-20
Top Features:
snapchat: 0.021363661672368948
google news: 0.020273928529063776
jio cinema: 0.019437754471972383
truecaller: 0.019220303746884767
wynk music: 0.018724523554299936
netflix: 0.01867566175845674
paytm money: 0.018445129788971407
uber: 0.01833814819094381
meesho: 0.018300544328650595
flipkart: 0.018033094372408347

Target Value: 21-25
Top Features:
snapchat: 0.021363661672368948
google news: 0.020273928529063776
jio cinema: 0.019437754471972383
truecaller: 0.019220303746884767
wynk music: 0.018724523554299936
netflix: 0.01867566175845674
paytm money: 0.018445129788971407
uber: 0.01833814819094381
meesho: 0.018300544328650595
flipkart: 0.018033094372408347

Target Value: 26-30
Top Features:
snapchat: 0.021363661672368948
google news: 0.020273928529063776
jio cinema: 0.019437754471972383
truecaller: 0.019220303746884767
wynk music: 0.018724523554299936
netflix: 0.01867566175845674
paytm money: 0.018445129788971407
uber: 0.01833814819094381
meesho: 0.0183005

### Way 2 (Category Level)

#### Transform Categories 

In [63]:
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_list_set,app_name,categories_l1,categories_l2,top_categories
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow,bookmyshow,entertainment,rest,entertainment
1,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube,youtube,tools,rest,rest
2,63ff28e9f22ead6bac895265,0,AVAILABLE,jio cinema,jio cinema,ott,rest,rest
3,63ff28e9f22ead6bac895265,0,AVAILABLE,google maps,google maps,tools,rest,rest


In [64]:
df_customer_mapped_v2 = df_main_app_category[~df_main_app_category['app_list_set'].isin(most_common_apps)]\
                        .groupby(['customer_id', 'age'])\
                        .agg(categories_l1 = ('categories_l1', lambda x: list(set(x))))\
                        .reset_index()

df_customer_mapped_v2.head(4)

,customer_id,age,categories_l1
0,5737c6adddbec2203f73316d,0,"[social, wearable, ott, driver_app, tools, tra..."
1,5737c6aeddbec2203f733176,34,"[finance_investment, delivery_grocery, social,..."
2,5737c6b0ddbec2203f733182,28,"[entertainment, social, ott, delivery_food, to..."
3,5737c6b1ddbec2203f73318b,33,"[travel_bookings, news, delivery_grocery, soci..."


In [65]:
v1_unique_categories = set(category for sublist in df_customer_mapped_v2['categories_l1'] for category in sublist)
v1_unique_categories_list = list(v1_unique_categories)
print("Count of unique apps:", len(v1_unique_categories))
sorted(v1_unique_categories)

Count of unique apps: 23


['dating',
 'delivery_food',
 'delivery_grocery',
 'devotional',
 'driver_app',
 'ecommerce',
 'educational',
 'entertainment',
 'fantasy_sports',
 'finance_investment',
 'finance_news',
 'finance_transactions',
 'gaming',
 'health',
 'news',
 'office',
 'ott',
 'social',
 'streaming_music',
 'tools',
 'travel_bookings',
 'travel_ridehailing',
 'wearable']

In [66]:
for category in v1_unique_categories:
    df_customer_mapped_v2[category] = df_customer_mapped_v2['categories_l1'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v2.head(1)

,customer_id,age,categories_l1,finance_investment,delivery_grocery,health,social,wearable,fantasy_sports,entertainment,office,gaming,news,ott,dating,streaming_music,travel_ridehailing,driver_app,ecommerce,finance_transactions,educational,devotional,travel_bookings,finance_news,delivery_food,tools
0,5737c6adddbec2203f73316d,0,"[social, wearable, ott, driver_app, tools, tra...",0,0,0,1,1,0,0,1,0,0,1,0,1,1,1,1,1,0,0,0,0,0,1


In [67]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['15-20', '21-25',  '26-30', '31-35', '36-45', '46-60']

# bins = [15, 20, 25, 35, 61]
# labels = ['15-20', '21-25',  '26-35', '36-60']

df_customer_mapped_v2['age_group'] = pd.cut(df_customer_mapped_v2['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [68]:
v1_main_list = ['customer_id', 'age', 'age_group' ,'categories_l1']
v1_unique_column_list = v1_main_list + v1_unique_categories_list

In [69]:
df_customer_mapped_v2 = df_customer_mapped_v2[v1_unique_column_list]
df_customer_mapped_v2.head(1)

,customer_id,age,age_group,categories_l1,finance_investment,delivery_grocery,health,social,wearable,fantasy_sports,entertainment,office,gaming,news,ott,dating,streaming_music,travel_ridehailing,driver_app,ecommerce,finance_transactions,educational,devotional,travel_bookings,finance_news,delivery_food,tools
0,5737c6adddbec2203f73316d,0,NaN,"[social, wearable, ott, driver_app, tools, tra...",0,0,0,1,1,0,0,1,0,0,1,0,1,1,1,1,1,0,0,0,0,0,1


In [70]:
df_customer_mapped_v2_age_w = df_customer_mapped_v2[~df_customer_mapped_v2['age_group'].isna()]
df_customer_mapped_v2_age_na = df_customer_mapped_v2[df_customer_mapped_v2['age_group'].isna()]
df_customer_mapped_v2_age_w.drop('categories_l1', axis=1, inplace=True)
df_customer_mapped_v2_age_na.drop('categories_l1', axis=1, inplace=True)

#### Model

In [71]:
# Step 1: Training the Model
X = df_customer_mapped_v2_age_w.drop(columns=['customer_id', 'age', 'age_group'])
y = df_customer_mapped_v2_age_w['age_group']

# Encode categorical variables if needed
label_encoder = LabelEncoder()
for col in X.columns:
    X[col] = label_encoder.fit_transform(X[col])
    
    
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)


model = RandomForestClassifier()  # Random Forest Classifier
model.fit(X_train, y_train)

RandomForestClassifier()

In [72]:
# Use the trained model to make predictions on the test set
y_pred = model.predict(X_test)

# Get unique target values
unique_classes = np.unique(y_test)

# Initialize dictionaries to store metrics for each target value
accuracy_dict = {}
precision_dict = {}
recall_dict = {}
f1_score_dict = {}

from sklearn.metrics import f1_score

# Calculate metrics for each target value
for class_name in unique_classes:
    # Filter predictions and true labels for the current target value
    y_true_class = (y_test == class_name)
    y_pred_class = (y_pred == class_name)

    # Calculate accuracy, recall, F1-score
    accuracy_dict[class_name] = accuracy_score(y_true_class, y_pred_class)
    precision_dict[class_name] = precision_score(y_true_class, y_pred_class)
    recall_dict[class_name] = recall_score(y_true_class, y_pred_class)
    f1_score_dict[class_name] = f1_score(y_true_class, y_pred_class)

In [73]:
for class_name in unique_classes:
    print(f"Target Value: {class_name}")
    print("Accuracy:", round(accuracy_dict[class_name] * 100.00, 2), "%")
    print("Precision:", round(precision_dict[class_name] * 100.00, 2), "%")
    print("Recall:", round(recall_dict[class_name] * 100.00, 2), "%")
    print("F1 Score:", round(f1_score_dict[class_name] * 100.00, 2), "%")
    print()

Target Value: 15-20
Accuracy: 94.87 %
Precision: 13.6 %
Recall: 2.85 %
F1 Score: 4.71 %

Target Value: 21-25
Accuracy: 68.62 %
Precision: 31.04 %
Recall: 27.03 %
F1 Score: 28.9 %

Target Value: 26-30
Accuracy: 48.17 %
Precision: 34.74 %
Recall: 65.1 %
F1 Score: 45.31 %

Target Value: 31-35
Accuracy: 74.41 %
Precision: 22.65 %
Recall: 10.54 %
F1 Score: 14.38 %

Target Value: 36-45
Accuracy: 81.34 %
Precision: 20.77 %
Recall: 9.3 %
F1 Score: 12.85 %

Target Value: 46-60
Accuracy: 95.72 %
Precision: 10.59 %
Recall: 1.71 %
F1 Score: 2.94 %



In [74]:
# Calculate the classification report
report = classification_report(y_test, y_pred, output_dict=True)

# Extract accuracy, precision, recall, and F1-score for each target value
overall_accuracy = report['accuracy']
overall_precision = report['macro avg']['precision']
overall_recall = report['macro avg']['recall']
overall_f1_score = report['macro avg']['f1-score']

print("Model Accuracy:", round(overall_accuracy*100.00, 2), "%")
print("Precision:", round(overall_precision*100.00, 2), "%")
print("Recall:", round(overall_recall*100.00, 2), "%")
print("F1 Score:", round(overall_f1_score*100.00, 2), "%")

Model Accuracy: 31.56 %
Precision: 22.23 %
Recall: 19.42 %
F1 Score: 18.18 %


#### Feature selection 

In [75]:
# Get feature importances
feature_importances = model.feature_importances_

# Map feature names to their importance scores
feature_importance_dict = dict(zip(X.columns, feature_importances))

# Iterate through each target value
for class_name in unique_classes:
    print(f"Target Value: {class_name}")
    
    # Filter the data for the current target value
    X_test_class = X_test[y_test == class_name]
    y_pred_class = y_pred[y_test == class_name]
    
    # Check if there are any instances of the current target value in the test set
    if len(X_test_class) > 0:
        # Get feature importances for the current target value
        feature_importance_class = {feature: importance for feature, importance in feature_importance_dict.items()}
        
        # Sort feature importances by importance score
        sorted_features = sorted(feature_importance_class.items(), key=lambda x: x[1], reverse=True)
        
        # Limit the number of top features displayed
        top_features = sorted_features[:10]  # Change the number here to display more or fewer top features
        
        # Print top features
        print("Top Features:")
        for feature, importance in top_features:
            print(f"{feature}: {importance}")
        print()
    else:
        print("No instances of this target value in the test set.")

Target Value: 15-20
Top Features:
delivery_grocery: 0.07912366782035739
tools: 0.07113393562562577
news: 0.07052087181608117
entertainment: 0.05923523068766095
driver_app: 0.05666562282171391
ott: 0.05665579151364759
office: 0.0550415592399491
finance_transactions: 0.05489299403236923
ecommerce: 0.05156294346022502
streaming_music: 0.04499387691459007

Target Value: 21-25
Top Features:
delivery_grocery: 0.07912366782035739
tools: 0.07113393562562577
news: 0.07052087181608117
entertainment: 0.05923523068766095
driver_app: 0.05666562282171391
ott: 0.05665579151364759
office: 0.0550415592399491
finance_transactions: 0.05489299403236923
ecommerce: 0.05156294346022502
streaming_music: 0.04499387691459007

Target Value: 26-30
Top Features:
delivery_grocery: 0.07912366782035739
tools: 0.07113393562562577
news: 0.07052087181608117
entertainment: 0.05923523068766095
driver_app: 0.05666562282171391
ott: 0.05665579151364759
office: 0.0550415592399491
finance_transactions: 0.05489299403236923
ecom

### Way 3 (App + Category Level)

#### Transform Categories 

In [76]:
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_list_set,app_name,categories_l1,categories_l2,top_categories
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow,bookmyshow,entertainment,rest,entertainment
1,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube,youtube,tools,rest,rest
2,63ff28e9f22ead6bac895265,0,AVAILABLE,jio cinema,jio cinema,ott,rest,rest
3,63ff28e9f22ead6bac895265,0,AVAILABLE,google maps,google maps,tools,rest,rest


In [77]:
df_customer_mapped_v3 = df_main_app_category[~df_main_app_category['app_list_set'].isin(most_common_apps)]\
                        .groupby(['customer_id', 'age'])\
                        .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))))\
                        .reset_index()

df_customer_mapped_v3.head(4)

,customer_id,age,app_list,categories_l1
0,5737c6adddbec2203f73316d,0,"[onedrive, disney+ hotstar, cred, rapido capta...","[social, wearable, ott, driver_app, tools, tra..."
1,5737c6aeddbec2203f733176,34,"[cred, groww, instagram, zomato, onedrive, mic...","[finance_investment, delivery_grocery, social,..."
2,5737c6b0ddbec2203f733182,28,"[zee5, ola driver app, disney+ hotstar, uber, ...","[entertainment, social, ott, delivery_food, to..."
3,5737c6b1ddbec2203f73318b,33,"[twitter, zee5, disney+ hotstar, uber, rapido ...","[travel_bookings, news, delivery_grocery, soci..."


In [78]:
v2_unique_categories = set(category for sublist in df_customer_mapped_v3['app_list'] for category in sublist)
v2_unique_categories_list = list(v2_unique_categories)
print("Count of unique apps:", len(v2_unique_categories))
sorted(v2_unique_categories)

Count of unique apps: 212


['aakash',
 'aastha tv',
 'accuweather',
 'adobe lightroom',
 'adobe scan',
 'aha',
 'airbnb',
 'aisle',
 'ajio',
 'alt balaji',
 'amazon music',
 'amazon prime video',
 'amazon shopping',
 'angel broking',
 'asana',
 'axis mobile',
 'bansal classes',
 'bigbasket',
 'blusmart',
 'booking.com',
 'bookmyshow',
 'borzo driver app',
 'brainly',
 'bumble',
 'byju',
 'caclubindia',
 'chegg study',
 'cisco',
 'cleartrip',
 'club factory',
 'coffee meets bagel',
 'colegeduniya',
 'collegedekho',
 'confluence',
 'course hero',
 'coursera',
 'cred',
 'cricplay',
 'diksha',
 'disney+ hotstar',
 'doubtnut',
 'driveu driver app',
 'dropbox',
 'dunzo',
 'dunzo driver app',
 'duolingo',
 'e-pg pathshala',
 'economic times',
 'edx',
 'embibe',
 'facebook',
 'feedly',
 'fiitjee',
 'file commander',
 'firstcry',
 'fitbit',
 'fiton',
 'flipboard',
 'flipkart',
 'fossil smartwatches',
 'freecharge',
 'gaana',
 'garmin connect',
 'geeks for geeks',
 'github',
 'goibibo',
 'goodreads',
 'google analytics',


In [79]:
for category in v2_unique_categories:
    df_customer_mapped_v3[category] = df_customer_mapped_v3['app_list'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v3.head(1)

,customer_id,age,app_list,categories_l1,myfitnesspal,onedrive,aisle,the indian express,aha,freecharge,garmin connect,feedly,hungama music,soundcloud,cred,mlb,microsoft lens,sonyliv,coursera,colegeduniya,zomato,zoom,google classroom,vidyakul,slack,...,redbus,amazon music,google sheets,uber eats,samsung galaxy wearable,dunzo,paytm money,chegg study,zomato driver app,vedantu,grofers,instagram,goibibo,ignou e-content,udemy,zoho mail,mx takatak,kite by zerodha,roposo,nba,driveu driver app,namma yatri,edx,stack overflow,indigo
0,5737c6adddbec2203f73316d,0,"[onedrive, disney+ hotstar, cred, rapido capta...","[social, wearable, ott, driver_app, tools, tra...",0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [81]:
v2a_unique_categories = set(category for sublist in df_customer_mapped_v3['categories_l1'] for category in sublist)
v2a_unique_categories_list = list(v2a_unique_categories)
print("Count of unique apps:", len(v2a_unique_categories))
sorted(v2a_unique_categories)

Count of unique apps: 23


['dating',
 'delivery_food',
 'delivery_grocery',
 'devotional',
 'driver_app',
 'ecommerce',
 'educational',
 'entertainment',
 'fantasy_sports',
 'finance_investment',
 'finance_news',
 'finance_transactions',
 'gaming',
 'health',
 'news',
 'office',
 'ott',
 'social',
 'streaming_music',
 'tools',
 'travel_bookings',
 'travel_ridehailing',
 'wearable']

In [82]:
for category in v2a_unique_categories:
    df_customer_mapped_v3[category] = df_customer_mapped_v3['categories_l1'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v3.head(1)

,customer_id,age,app_list,categories_l1,myfitnesspal,onedrive,aisle,the indian express,aha,freecharge,garmin connect,feedly,hungama music,soundcloud,cred,mlb,microsoft lens,sonyliv,coursera,colegeduniya,zomato,zoom,google classroom,vidyakul,slack,...,stack overflow,indigo,finance_investment,delivery_grocery,health,social,wearable,fantasy_sports,entertainment,office,gaming,news,ott,dating,streaming_music,travel_ridehailing,driver_app,ecommerce,finance_transactions,educational,devotional,travel_bookings,finance_news,delivery_food,tools
0,5737c6adddbec2203f73316d,0,"[onedrive, disney+ hotstar, cred, rapido capta...","[social, wearable, ott, driver_app, tools, tra...",0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,0,0,1,0,0,1,0,1,1,1,1,1,0,0,0,0,0,1


In [83]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['15-20', '21-25',  '26-30', '31-35', '36-45', '46-60']

# bins = [15, 20, 25, 35, 61]
# labels = ['15-20', '21-25',  '26-35', '36-60']

df_customer_mapped_v3['age_group'] = pd.cut(df_customer_mapped_v3['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [84]:
v2_main_list = ['customer_id', 'age', 'age_group' , 'app_list', 'categories_l1']
v2_unique_column_list = v2_main_list + v2_unique_categories_list + v2a_unique_categories_list

In [85]:
df_customer_mapped_v3 = df_customer_mapped_v3[v2_unique_column_list]
df_customer_mapped_v3.head(1)

,customer_id,age,age_group,app_list,categories_l1,myfitnesspal,onedrive,aisle,the indian express,aha,freecharge,garmin connect,feedly,hungama music,soundcloud,cred,mlb,microsoft lens,sonyliv,coursera,colegeduniya,zomato,zoom,google classroom,vidyakul,...,stack overflow,indigo,finance_investment,delivery_grocery,health,social,wearable,fantasy_sports,entertainment,office,gaming,news,ott,dating,streaming_music,travel_ridehailing,driver_app,ecommerce,finance_transactions,educational,devotional,travel_bookings,finance_news,delivery_food,tools
0,5737c6adddbec2203f73316d,0,NaN,"[onedrive, disney+ hotstar, cred, rapido capta...","[social, wearable, ott, driver_app, tools, tra...",0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,1,1,0,0,1,0,0,1,0,1,1,1,1,1,0,0,0,0,0,1


In [87]:
df_customer_mapped_v3_age_w = df_customer_mapped_v3[~df_customer_mapped_v3['age_group'].isna()]
df_customer_mapped_v3_age_na = df_customer_mapped_v3[df_customer_mapped_v3['age_group'].isna()]
df_customer_mapped_v3_age_w.drop(columns=['categories_l1', 'app_list'], axis=1, inplace=True)
df_customer_mapped_v3_age_na.drop(columns=['categories_l1', 'app_list'], axis=1, inplace=True)

#### Model

In [88]:
# Step 1: Training the Model
X = df_customer_mapped_v3_age_w.drop(columns=['customer_id', 'age', 'age_group'])
y = df_customer_mapped_v3_age_w['age_group']

# Encode categorical variables if needed
label_encoder = LabelEncoder()
for col in X.columns:
    X[col] = label_encoder.fit_transform(X[col])
    
    
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)


model = RandomForestClassifier()  # Random Forest Classifier
model.fit(X_train, y_train)

RandomForestClassifier()

In [89]:
# Use the trained model to make predictions on the test set
y_pred = model.predict(X_test)

# Get unique target values
unique_classes = np.unique(y_test)

# Initialize dictionaries to store metrics for each target value
accuracy_dict = {}
precision_dict = {}
recall_dict = {}
f1_score_dict = {}

from sklearn.metrics import f1_score

# Calculate metrics for each target value
for class_name in unique_classes:
    # Filter predictions and true labels for the current target value
    y_true_class = (y_test == class_name)
    y_pred_class = (y_pred == class_name)

    # Calculate accuracy, recall, F1-score
    accuracy_dict[class_name] = accuracy_score(y_true_class, y_pred_class)
    precision_dict[class_name] = precision_score(y_true_class, y_pred_class)
    recall_dict[class_name] = recall_score(y_true_class, y_pred_class)
    f1_score_dict[class_name] = f1_score(y_true_class, y_pred_class)

In [90]:
for class_name in unique_classes:
    print(f"Target Value: {class_name}")
    print("Accuracy:", round(accuracy_dict[class_name] * 100.00, 2), "%")
    print("Precision:", round(precision_dict[class_name] * 100.00, 2), "%")
    print("Recall:", round(recall_dict[class_name] * 100.00, 2), "%")
    print("F1 Score:", round(f1_score_dict[class_name] * 100.00, 2), "%")
    print()

Target Value: 15-20
Accuracy: 95.33 %
Precision: 28.14 %
Recall: 3.2 %
F1 Score: 5.75 %

Target Value: 21-25
Accuracy: 75.48 %
Precision: 47.99 %
Recall: 47.02 %
F1 Score: 47.5 %

Target Value: 26-30
Accuracy: 56.66 %
Precision: 40.46 %
Recall: 66.69 %
F1 Score: 50.36 %

Target Value: 31-35
Accuracy: 75.23 %
Precision: 33.31 %
Recall: 21.35 %
F1 Score: 26.02 %

Target Value: 36-45
Accuracy: 82.78 %
Precision: 35.9 %
Recall: 20.93 %
F1 Score: 26.44 %

Target Value: 46-60
Accuracy: 96.01 %
Precision: 22.13 %
Recall: 1.97 %
F1 Score: 3.62 %



In [92]:
# Calculate the classification report
report = classification_report(y_test, y_pred, output_dict=True)

# Extract accuracy, precision, recall, and F1-score for each target value
overall_accuracy = report['accuracy']
overall_precision = report['macro avg']['precision']
overall_recall = report['macro avg']['recall']
overall_f1_score = report['macro avg']['f1-score']

print("Model Accuracy:", round(overall_accuracy*100.00, 2), "%")
print("Precision:", round(overall_precision*100.00, 2), "%")
print("Recall:", round(overall_recall*100.00, 2), "%")
print("F1 Score:", round(overall_f1_score*100.00, 2), "%")

Model Accuracy: 40.75 %
Precision: 34.66 %
Recall: 26.86 %
F1 Score: 26.62 %


#### Feature selection 

In [91]:
# Get feature importances
feature_importances = model.feature_importances_

# Map feature names to their importance scores
feature_importance_dict = dict(zip(X.columns, feature_importances))

# Iterate through each target value
for class_name in unique_classes:
    print(f"Target Value: {class_name}")
    
    # Filter the data for the current target value
    X_test_class = X_test[y_test == class_name]
    y_pred_class = y_pred[y_test == class_name]
    
    # Check if there are any instances of the current target value in the test set
    if len(X_test_class) > 0:
        # Get feature importances for the current target value
        feature_importance_class = {feature: importance for feature, importance in feature_importance_dict.items()}
        
        # Sort feature importances by importance score
        sorted_features = sorted(feature_importance_class.items(), key=lambda x: x[1], reverse=True)
        
        # Limit the number of top features displayed
        top_features = sorted_features[:10]  # Change the number here to display more or fewer top features
        
        # Print top features
        print("Top Features:")
        for feature, importance in top_features:
            print(f"{feature}: {importance}")
        print()
    else:
        print("No instances of this target value in the test set.")

Target Value: 15-20
Top Features:
snapchat: 0.020881734571361284
jio cinema: 0.01591845143661935
meesho: 0.015450758412904194
netflix: 0.015432618046912467
disney+ hotstar: 0.014610420168976799
truecaller: 0.014585712941747571
uber: 0.01418979170233151
wynk music: 0.014104204270317854
linkedin: 0.014066688814435836
telegram: 0.014005371316079802

Target Value: 21-25
Top Features:
snapchat: 0.020881734571361284
jio cinema: 0.01591845143661935
meesho: 0.015450758412904194
netflix: 0.015432618046912467
disney+ hotstar: 0.014610420168976799
truecaller: 0.014585712941747571
uber: 0.01418979170233151
wynk music: 0.014104204270317854
linkedin: 0.014066688814435836
telegram: 0.014005371316079802

Target Value: 26-30
Top Features:
snapchat: 0.020881734571361284
jio cinema: 0.01591845143661935
meesho: 0.015450758412904194
netflix: 0.015432618046912467
disney+ hotstar: 0.014610420168976799
truecaller: 0.014585712941747571
uber: 0.01418979170233151
wynk music: 0.014104204270317854
linkedin: 0.0140

### Way 4 (Top App)

#### Transform Categories 

In [93]:
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_list_set,app_name,categories_l1,categories_l2,top_categories
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow,bookmyshow,entertainment,rest,entertainment
1,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube,youtube,tools,rest,rest
2,63ff28e9f22ead6bac895265,0,AVAILABLE,jio cinema,jio cinema,ott,rest,rest
3,63ff28e9f22ead6bac895265,0,AVAILABLE,google maps,google maps,tools,rest,rest


In [94]:
df_customer_mapped_v4 = df_main_app_category[~df_main_app_category['top_categories'].isin(['rest'])]\
                        .groupby(['customer_id', 'age'])\
                        .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))))\
                        .reset_index()

df_customer_mapped_v4.head(4)

,customer_id,age,app_list,categories_l1
0,5737c6adddbec2203f73316d,0,"[rapido captain, instagram, onedrive, linkedin...","[social, driver_app, travel_ridehailing, strea..."
1,5737c6aeddbec2203f733176,34,"[groww, instagram, zomato, onedrive, microsoft...","[finance_investment, social, delivery_food, tr..."
2,5737c6b0ddbec2203f733182,28,"[telegram, instagram, rapido captain, zomato, ...","[entertainment, social, delivery_food, travel_..."
3,5737c6b1ddbec2203f73318b,33,"[telegram, instagram, goibibo, twitter, zomato...","[travel_bookings, office, social, delivery_foo..."


In [95]:
v3_unique_categories = set(category for sublist in df_customer_mapped_v4['app_list'] for category in sublist)
v3_unique_categories_list = list(v3_unique_categories)
print("Count of unique apps:", len(v3_unique_categories))
sorted(v3_unique_categories)

Count of unique apps: 109


['aakash',
 'airbnb',
 'aisle',
 'amazon music',
 'angel broking',
 'asana',
 'blusmart',
 'booking.com',
 'bookmyshow',
 'borzo driver app',
 'bumble',
 'chegg study',
 'cisco',
 'cleartrip',
 'coffee meets bagel',
 'confluence',
 'diksha',
 'driveu driver app',
 'dropbox',
 'dunzo',
 'dunzo driver app',
 'e-pg pathshala',
 'facebook',
 'flipboard',
 'gaana',
 'github',
 'goibibo',
 'google analytics',
 'google authenticator',
 'google classroom',
 'groww',
 'happn',
 'hdfc mobile banking',
 'hike',
 'hinge',
 'hungama music',
 'in drive',
 'indigo',
 'instagram',
 'intune  company portal',
 'jira',
 'jugnoo',
 'kayak',
 'kite by zerodha',
 'line',
 'linkedin',
 'ludo king',
 'messenger',
 'microsoft 365',
 'microsoft math solver',
 'microsoft teams',
 'miro',
 'moodle',
 'ms authenticator',
 'mx takatak',
 'mycbseguide',
 'namma partner',
 'namma yatri',
 'nishtha',
 'nps',
 'okcupid',
 'ola',
 'ola driver app',
 'onedrive',
 'outlook',
 'physics wallah',
 'pinterest',
 'pocket',
 'p

In [96]:
for category in v3_unique_categories:
    df_customer_mapped_v4[category] = df_customer_mapped_v4['app_list'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v4.head(1)

,customer_id,age,app_list,categories_l1,onedrive,aisle,hungama music,soundcloud,zomato,zoom,google classroom,vidyakul,slack,wynk music,coffee meets bagel,cleartrip,simplilearn,trulymadly,viber,snapchat,microsoft teams,trivago,physics wallah,nishtha,whatsapp,...,github,miro,tripadvisor,pinterest,angel broking,dunzo driver app,groww,intune company portal,tinder,redbus,amazon music,uber eats,dunzo,chegg study,zomato driver app,vedantu,instagram,goibibo,zoho mail,mx takatak,kite by zerodha,roposo,driveu driver app,namma yatri,indigo
0,5737c6adddbec2203f73316d,0,"[rapido captain, instagram, onedrive, linkedin...","[social, driver_app, travel_ridehailing, strea...",1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [97]:
v3a_unique_categories = set(category for sublist in df_customer_mapped_v4['categories_l1'] for category in sublist)
v3a_unique_categories_list = list(v3a_unique_categories)
print("Count of unique apps:", len(v3a_unique_categories))
sorted(v3a_unique_categories)

Count of unique apps: 12


['dating',
 'delivery_food',
 'driver_app',
 'educational',
 'entertainment',
 'finance_investment',
 'gaming',
 'office',
 'social',
 'streaming_music',
 'travel_bookings',
 'travel_ridehailing']

In [98]:
for category in v3a_unique_categories:
    df_customer_mapped_v4[category] = df_customer_mapped_v4['categories_l1'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v4.head(1)

,customer_id,age,app_list,categories_l1,onedrive,aisle,hungama music,soundcloud,zomato,zoom,google classroom,vidyakul,slack,wynk music,coffee meets bagel,cleartrip,simplilearn,trulymadly,viber,snapchat,microsoft teams,trivago,physics wallah,nishtha,whatsapp,...,dunzo,chegg study,zomato driver app,vedantu,instagram,goibibo,zoho mail,mx takatak,kite by zerodha,roposo,driveu driver app,namma yatri,indigo,educational,travel_bookings,office,finance_investment,entertainment,social,delivery_food,travel_ridehailing,dating,streaming_music,driver_app,gaming
0,5737c6adddbec2203f73316d,0,"[rapido captain, instagram, onedrive, linkedin...","[social, driver_app, travel_ridehailing, strea...",1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,1,1,0


In [99]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['15-20', '21-25',  '26-30', '31-35', '36-45', '46-60']

# bins = [15, 20, 25, 35, 61]
# labels = ['15-20', '21-25',  '26-35', '36-60']

df_customer_mapped_v4['age_group'] = pd.cut(df_customer_mapped_v4['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [100]:
v3_main_list = ['customer_id', 'age', 'age_group' , 'app_list', 'categories_l1']
v3_unique_column_list = v3_main_list + v3_unique_categories_list + v3a_unique_categories_list

In [101]:
df_customer_mapped_v4 = df_customer_mapped_v4[v3_unique_column_list]
df_customer_mapped_v4.head(1)

,customer_id,age,age_group,app_list,categories_l1,onedrive,aisle,hungama music,soundcloud,zomato,zoom,google classroom,vidyakul,slack,wynk music,coffee meets bagel,cleartrip,simplilearn,trulymadly,viber,snapchat,microsoft teams,trivago,physics wallah,nishtha,...,dunzo,chegg study,zomato driver app,vedantu,instagram,goibibo,zoho mail,mx takatak,kite by zerodha,roposo,driveu driver app,namma yatri,indigo,educational,travel_bookings,office,finance_investment,entertainment,social,delivery_food,travel_ridehailing,dating,streaming_music,driver_app,gaming
0,5737c6adddbec2203f73316d,0,NaN,"[rapido captain, instagram, onedrive, linkedin...","[social, driver_app, travel_ridehailing, strea...",1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,1,1,0


In [102]:
df_customer_mapped_v4_age_w = df_customer_mapped_v4[~df_customer_mapped_v4['age_group'].isna()]
df_customer_mapped_v4_age_na = df_customer_mapped_v4[df_customer_mapped_v4['age_group'].isna()]
df_customer_mapped_v4_age_w.drop(columns=['categories_l1', 'app_list'], axis=1, inplace=True)
df_customer_mapped_v4_age_na.drop(columns=['categories_l1', 'app_list'], axis=1, inplace=True)

#### Model

In [103]:
# Step 1: Training the Model
X = df_customer_mapped_v4_age_w.drop(columns=['customer_id', 'age', 'age_group'])
y = df_customer_mapped_v4_age_w['age_group']

# Encode categorical variables if needed
label_encoder = LabelEncoder()
for col in X.columns:
    X[col] = label_encoder.fit_transform(X[col])
    
    
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)


model = RandomForestClassifier()  # Random Forest Classifier
model.fit(X_train, y_train)

RandomForestClassifier()

In [104]:
# Use the trained model to make predictions on the test set
y_pred = model.predict(X_test)

# Get unique target values
unique_classes = np.unique(y_test)

# Initialize dictionaries to store metrics for each target value
accuracy_dict = {}
precision_dict = {}
recall_dict = {}
f1_score_dict = {}

from sklearn.metrics import f1_score

# Calculate metrics for each target value
for class_name in unique_classes:
    # Filter predictions and true labels for the current target value
    y_true_class = (y_test == class_name)
    y_pred_class = (y_pred == class_name)

    # Calculate accuracy, recall, F1-score
    accuracy_dict[class_name] = accuracy_score(y_true_class, y_pred_class)
    precision_dict[class_name] = precision_score(y_true_class, y_pred_class)
    recall_dict[class_name] = recall_score(y_true_class, y_pred_class)
    f1_score_dict[class_name] = f1_score(y_true_class, y_pred_class)

In [105]:
for class_name in unique_classes:
    print(f"Target Value: {class_name}")
    print("Accuracy:", round(accuracy_dict[class_name] * 100.00, 2), "%")
    print("Precision:", round(precision_dict[class_name] * 100.00, 2), "%")
    print("Recall:", round(recall_dict[class_name] * 100.00, 2), "%")
    print("F1 Score:", round(f1_score_dict[class_name] * 100.00, 2), "%")
    print()

Target Value: 15-20
Accuracy: 94.86 %
Precision: 16.91 %
Recall: 4.31 %
F1 Score: 6.86 %

Target Value: 21-25
Accuracy: 73.43 %
Precision: 43.35 %
Recall: 43.25 %
F1 Score: 43.3 %

Target Value: 26-30
Accuracy: 56.56 %
Precision: 39.41 %
Recall: 58.44 %
F1 Score: 47.07 %

Target Value: 31-35
Accuracy: 73.56 %
Precision: 29.86 %
Recall: 21.93 %
F1 Score: 25.29 %

Target Value: 36-45
Accuracy: 81.12 %
Precision: 31.14 %
Recall: 22.07 %
F1 Score: 25.83 %

Target Value: 46-60
Accuracy: 95.61 %
Precision: 16.88 %
Recall: 4.07 %
F1 Score: 6.56 %



In [106]:
# Calculate the classification report
report = classification_report(y_test, y_pred, output_dict=True)

# Extract accuracy, precision, recall, and F1-score for each target value
overall_accuracy = report['accuracy']
overall_precision = report['macro avg']['precision']
overall_recall = report['macro avg']['recall']
overall_f1_score = report['macro avg']['f1-score']

print("Model Accuracy:", round(overall_accuracy*100.00, 2), "%")
print("Precision:", round(overall_precision*100.00, 2), "%")
print("Recall:", round(overall_recall*100.00, 2), "%")
print("F1 Score:", round(overall_f1_score*100.00, 2), "%")

Model Accuracy: 37.57 %
Precision: 29.59 %
Recall: 25.68 %
F1 Score: 25.82 %


#### Feature selection 

In [107]:
# Get feature importances
feature_importances = model.feature_importances_

# Map feature names to their importance scores
feature_importance_dict = dict(zip(X.columns, feature_importances))

# Iterate through each target value
for class_name in unique_classes:
    print(f"Target Value: {class_name}")
    
    # Filter the data for the current target value
    X_test_class = X_test[y_test == class_name]
    y_pred_class = y_pred[y_test == class_name]
    
    # Check if there are any instances of the current target value in the test set
    if len(X_test_class) > 0:
        # Get feature importances for the current target value
        feature_importance_class = {feature: importance for feature, importance in feature_importance_dict.items()}
        
        # Sort feature importances by importance score
        sorted_features = sorted(feature_importance_class.items(), key=lambda x: x[1], reverse=True)
        
        # Limit the number of top features displayed
        top_features = sorted_features[:10]  # Change the number here to display more or fewer top features
        
        # Print top features
        print("Top Features:")
        for feature, importance in top_features:
            print(f"{feature}: {importance}")
        print()
    else:
        print("No instances of this target value in the test set.")

Target Value: 15-20
Top Features:
uber: 0.029298831327649905
snapchat: 0.027727339367662668
linkedin: 0.02740177301837956
wynk music: 0.027233089117715527
swiggy: 0.026589000010282686
twitter: 0.025173730479118386
namma yatri: 0.024715079070848245
telegram: 0.023606835060760917
pinterest: 0.021334762191000053
microsoft teams: 0.02082731960566478

Target Value: 21-25
Top Features:
uber: 0.029298831327649905
snapchat: 0.027727339367662668
linkedin: 0.02740177301837956
wynk music: 0.027233089117715527
swiggy: 0.026589000010282686
twitter: 0.025173730479118386
namma yatri: 0.024715079070848245
telegram: 0.023606835060760917
pinterest: 0.021334762191000053
microsoft teams: 0.02082731960566478

Target Value: 26-30
Top Features:
uber: 0.029298831327649905
snapchat: 0.027727339367662668
linkedin: 0.02740177301837956
wynk music: 0.027233089117715527
swiggy: 0.026589000010282686
twitter: 0.025173730479118386
namma yatri: 0.024715079070848245
telegram: 0.023606835060760917
pinterest: 0.0213347621

### Next classify step 

In [52]:
# Keep customer_id &  values for reference
customer_ids = df_customer_mapped_age_na['customer_id']

In [53]:
# Step 2: Predicting with the Model
for col in df_customer_mapped_age_na.columns:
    df_customer_mapped_age_na[col] = label_encoder.fit_transform(df_customer_mapped_age_na[col])

    
# Predict age_group for df_customer_mapped_age_na
df_customer_mapped_age_na['predicted_age_group'] = model.predict(df_customer_mapped_age_na\
                                                                 .drop(columns=['customer_id','age','age_group']))
df_customer_mapped_age_na['age_group'] = df_customer_mapped_age_na['predicted_age_group']
df_customer_mapped_age_na = df_customer_mapped_age_na.drop('predicted_age_group', axis=1)

# Add customer_id column back 
df_customer_mapped_age_na['customer_id'] = customer_ids

In [54]:
df_customer_mapped_age_w.columns

Index(['customer_id', 'age', 'age_group', 'physics wallah', 'linkedin', 'zoom',
       'spotify', 'yatra', 'quackquack', 'porter driver app',
       ...
       'swiggy driver app', 'nishtha', 'ola driver app', 'chegg study',
       'trello', 'redbus', 'sharechat', 'intune  company portal', 'uber eats',
       'ludo king'],
      dtype='object', length=112)

In [55]:
df_customer_mapped_age_na.columns

Index(['customer_id', 'age', 'age_group', 'physics wallah', 'linkedin', 'zoom',
       'spotify', 'yatra', 'quackquack', 'porter driver app',
       ...
       'swiggy driver app', 'nishtha', 'ola driver app', 'chegg study',
       'trello', 'redbus', 'sharechat', 'intune  company portal', 'uber eats',
       'ludo king'],
      dtype='object', length=112)

In [56]:
df_customer_mapped_age_w.head(1)

,customer_id,age,age_group,physics wallah,linkedin,zoom,spotify,yatra,quackquack,porter driver app,cleartrip,namma yatri,ms authenticator,amazon music,zoho meeting,quick ride,nps,driveu driver app,cisco,okcupid,vidyakul,trivago,shadowfax driver app,swiggy,telegram,...,airbnb,borzo driver app,facebook,blusmart,indigo,zomato driver app,soundcloud,microsoft math solver,mx takatak,groww,pocket,tripadvisor,goibibo,jugnoo,hungama music,swiggy driver app,nishtha,ola driver app,chegg study,trello,redbus,sharechat,intune company portal,uber eats,ludo king
1,5737c6aeddbec2203f733176,34,26-35,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [57]:
df_customer_mapped_age_na.head(1)

,customer_id,age,age_group,physics wallah,linkedin,zoom,spotify,yatra,quackquack,porter driver app,cleartrip,namma yatri,ms authenticator,amazon music,zoho meeting,quick ride,nps,driveu driver app,cisco,okcupid,vidyakul,trivago,shadowfax driver app,swiggy,telegram,...,airbnb,borzo driver app,facebook,blusmart,indigo,zomato driver app,soundcloud,microsoft math solver,mx takatak,groww,pocket,tripadvisor,goibibo,jugnoo,hungama music,swiggy driver app,nishtha,ola driver app,chegg study,trello,redbus,sharechat,intune company portal,uber eats,ludo king
0,5737c6adddbec2203f73316d,0,26-35,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [58]:
df_customer_mapped_age_w.groupby(['age_group']).agg(customer = ('customer_id','nunique'))

,customer
age_group,
15-20,27525
21-25,146648
26-35,332685
36-60,116084


In [59]:
df_customer_mapped_age_na.groupby(['age_group']).agg(customer = ('customer_id','nunique'))

,customer
age_group,
15-20,22665
21-25,265993
26-35,829874
36-60,135537


In [60]:
df_classified_data = pd.concat([df_customer_mapped_age_w, df_customer_mapped_age_na])

# If you want to reset the index after concatenating
df_classified_data.reset_index(drop=True, inplace=True)
df_classified_data = df_classified_data[['customer_id', 'age_group']]


df_classified_data = pd.merge(df_customer_mapped[['customer_id', 'top_categories']], 
                              df_classified_data,
                              how='left',
                              on='customer_id')

df_classified_data.head(2)

,customer_id,top_categories,age_group
0,5737c6adddbec2203f73316d,"[travel_ridehailing, office, streaming_music, ...",26-35
1,5737c6aeddbec2203f733176,"[travel_ridehailing, office, finance_investmen...",26-35


In [61]:
df_classified_data.age_group.unique()

array(['26-35', '36-60', '15-20', '21-25'], dtype=object)

In [62]:
unique_categories2 = set(category for sublist in df_classified_data['top_categories'] for category in sublist)
unique_categories2 = list(unique_categories2)
print("Count of unique category:", len(unique_categories2))
sorted(unique_categories2)

Count of unique category: 14


['core_education',
 'core_office',
 'dating',
 'delivery_food',
 'driver_app',
 'entertainment',
 'finance_investment',
 'gaming',
 'office',
 'rest',
 'social',
 'streaming_music',
 'travel_bookings',
 'travel_ridehailing']

In [63]:
for category in unique_categories2:
    df_classified_data[category] = df_classified_data['top_categories'].apply(lambda x: 1 if category in x else 0) 
    
df_classified_data.drop('top_categories', axis=1, inplace=True)    
df_classified_data.head(1)

,customer_id,age_group,travel_ridehailing,office,finance_investment,rest,core_office,gaming,dating,delivery_food,core_education,streaming_music,social,travel_bookings,driver_app,entertainment
0,5737c6adddbec2203f73316d,26-35,1,1,0,0,0,0,0,0,0,1,1,0,1,0


## Triangulation 2 - Customer use-case

In [64]:
df_all_customer.columns

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 'ltr_segment',
       'life_time_rides', 'life_time_stage', 'service_affinity', 'age',
       'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
       'second_half_pickup_primary', 'second_half_pickup_secondary',
       'app_list_set', 'age_presence', 'appo_presence'],
      dtype='object')

In [65]:
df_all_customer[['customer_id', # 'morning_fe', 'evening_fe', 
                 'morning_rr', 'evening_rr',
                 'first_half_drop_primary', 'first_half_drop_secondary',
                 'second_half_pickup_primary', 'second_half_pickup_secondary']]

,customer_id,morning_rr,evening_rr,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,63ff28e9f22ead6bac895265,1,10,leisure,NaN,residential,NaN
1,5cb7f15554bc7263ff3f4e73,9,4,health_and_personal,place_of_worship,office,residential
2,62428c959d4b75f09cb3a511,72,27,household_needs,transit_station,household_needs,transit_station
3,6222fa122b0c465e76b47c76,22,11,NaN,NaN,NaN,NaN
4,5fbaaf067e483f4a13fd9b9e,9,32,transit_station,office,residential,health_and_personal
...,...,...,...,...,...,...,...
2766888,65e6a82fd8310c2a5f675b0d,1,0,office,NaN,NaN,NaN
2766889,65eb2f7ff2a98f9872e52218,0,0,NaN,NaN,NaN,NaN
2766890,62b13e3fc6b74847db76db77,1,1,residential,NaN,residential,NaN
2766891,649bb175e318464c7066d22d,0,0,NaN,NaN,NaN,NaN


In [66]:
df_triangulation_2 = df_all_customer[(~df_all_customer['first_half_drop_primary'].isna() 
                                      | 
                                      ~df_all_customer['first_half_drop_secondary'].isna()
                                      |
                                      ~df_all_customer['second_half_pickup_primary'].isna() 
                                      | 
                                      ~df_all_customer['second_half_pickup_secondary'].isna()
                                     )]\
                                    [['customer_id', 'age', 'morning_rr', 'evening_rr',
                                      'first_half_drop_primary','first_half_drop_secondary',
                                      'second_half_pickup_primary', 'second_half_pickup_secondary']]

df_triangulation_2.head(5)

,customer_id,age,morning_rr,evening_rr,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,63ff28e9f22ead6bac895265,0,1,10,leisure,NaN,residential,NaN
1,5cb7f15554bc7263ff3f4e73,28,9,4,health_and_personal,place_of_worship,office,residential
2,62428c959d4b75f09cb3a511,0,72,27,household_needs,transit_station,household_needs,transit_station
4,5fbaaf067e483f4a13fd9b9e,28,9,32,transit_station,office,residential,health_and_personal
5,64d0d9ebc810cc925f36ceea,0,19,53,residential,transit_station,residential,educational


In [67]:
df_triangulation_2['primary_use_case'] = np.where(df_triangulation_2['morning_rr'] > df_triangulation_2['evening_rr'],
                                             df_triangulation_2['first_half_drop_primary'],
                                             np.where(df_triangulation_2['morning_rr'] < df_triangulation_2['evening_rr'],
                                                      df_triangulation_2['second_half_pickup_primary'],
                                                      df_triangulation_2['first_half_drop_primary']))
df_triangulation_2['secondary_use_case'] = np.where(df_triangulation_2['morning_rr'] > df_triangulation_2['evening_rr'],
                                                 df_triangulation_2['first_half_drop_secondary'],
                                                 np.where(df_triangulation_2['morning_rr'] < df_triangulation_2['evening_rr'],
                                                          df_triangulation_2['second_half_pickup_secondary'],
                                                          df_triangulation_2['first_half_drop_secondary']))

df_use_case_data = df_triangulation_2[['customer_id', 'primary_use_case', 'secondary_use_case']]
df_use_case_data

,customer_id,primary_use_case,secondary_use_case
0,63ff28e9f22ead6bac895265,residential,NaN
1,5cb7f15554bc7263ff3f4e73,health_and_personal,place_of_worship
2,62428c959d4b75f09cb3a511,household_needs,transit_station
4,5fbaaf067e483f4a13fd9b9e,residential,health_and_personal
5,64d0d9ebc810cc925f36ceea,residential,educational
...,...,...,...
2766884,6441374fce29286c60239f57,household_needs,NaN
2766885,635d3a958caf932c8bb2383f,transit_station,residential
2766886,5ddb98796ffaa931412b03b6,food,transit_station
2766888,65e6a82fd8310c2a5f675b0d,office,NaN


## Triangulation 3 - Customer metric

In [68]:
df_all_customer.columns

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 'ltr_segment',
       'life_time_rides', 'life_time_stage', 'service_affinity', 'age',
       'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
       'second_half_pickup_primary', 'second_half_pickup_secondary',
       'app_list_set', 'age_presence', 'appo_presence'],
      dtype='object')

In [69]:
df_all_customer[['customer_id', 'fe_count', 'rr_count', 'net_count',
                 'ltr_segment', 'life_time_rides', 'life_time_stage', 'service_affinity']]

,customer_id,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,service_affinity
0,63ff28e9f22ead6bac895265,20,12,8,PHH,37.0,HOOK,ONLY_LINK
1,5cb7f15554bc7263ff3f4e73,61,23,17,PHH,95.0,COMMITTED,ONLY_AUTO
2,62428c959d4b75f09cb3a511,157,103,30,PHH,248.0,COMMITTED,ONLY_AUTO
3,6222fa122b0c465e76b47c76,100,33,9,PHH,76.0,COMMITTED,ONLY_LINK
4,5fbaaf067e483f4a13fd9b9e,85,45,15,PHH,87.0,COMMITTED,ONLY_AUTO
...,...,...,...,...,...,...,...,...
2766888,65e6a82fd8310c2a5f675b0d,9,1,1,HH,1.0,DORMANT,ONLY_LINK
2766889,65eb2f7ff2a98f9872e52218,4,0,0,NaN,NaN,NaN,NaN
2766890,62b13e3fc6b74847db76db77,7,3,2,PHH,41.0,HOOK,ONLY_AUTO
2766891,649bb175e318464c7066d22d,2,1,1,HH,1.0,HANDHOLDING,ONLY_AUTO


In [70]:
df_triangulation_3 = df_all_customer[['customer_id', 'net_count', 'ltr_segment', 
                                      'life_time_rides', 'life_time_stage', 'service_affinity']]
df_triangulation_3.rename(columns = {'net_count' :'rpc_monthly'}, inplace = True)
df_metric_data = df_triangulation_3
df_metric_data

,customer_id,rpc_monthly,ltr_segment,life_time_rides,life_time_stage,service_affinity
0,63ff28e9f22ead6bac895265,8,PHH,37.0,HOOK,ONLY_LINK
1,5cb7f15554bc7263ff3f4e73,17,PHH,95.0,COMMITTED,ONLY_AUTO
2,62428c959d4b75f09cb3a511,30,PHH,248.0,COMMITTED,ONLY_AUTO
3,6222fa122b0c465e76b47c76,9,PHH,76.0,COMMITTED,ONLY_LINK
4,5fbaaf067e483f4a13fd9b9e,15,PHH,87.0,COMMITTED,ONLY_AUTO
...,...,...,...,...,...,...
2766888,65e6a82fd8310c2a5f675b0d,1,HH,1.0,DORMANT,ONLY_LINK
2766889,65eb2f7ff2a98f9872e52218,0,NaN,NaN,NaN,NaN
2766890,62b13e3fc6b74847db76db77,2,PHH,41.0,HOOK,ONLY_AUTO
2766891,649bb175e318464c7066d22d,1,HH,1.0,HANDHOLDING,ONLY_AUTO


# Triangulation

In [71]:
df_all_customer.head(1)

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set,age_presence,appo_presence
0,android,63ff28e9f22ead6bac895265,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,0,MALE,leisure,NaN,residential,NaN,"[bookmyshow, youtube, jio cinema, google maps,...",NA,AVAILABLE


In [72]:
df_classified_data.head(1)

,customer_id,age_group,travel_ridehailing,office,finance_investment,rest,core_office,gaming,dating,delivery_food,core_education,streaming_music,social,travel_bookings,driver_app,entertainment
0,5737c6adddbec2203f73316d,26-35,1,1,0,0,0,0,0,0,0,1,1,0,1,0


In [73]:
df_use_case_data.head(1)

,customer_id,primary_use_case,secondary_use_case
0,63ff28e9f22ead6bac895265,residential,NaN


In [74]:
df_metric_data.head(1)

,customer_id,rpc_monthly,ltr_segment,life_time_rides,life_time_stage,service_affinity
0,63ff28e9f22ead6bac895265,8,PHH,37.0,HOOK,ONLY_LINK


## Triangulation of 3 data sources

In [109]:
df_consolidate=df_all_customer[df_all_customer['platform'].isin(['android'])][['platform', 'customer_id', 'age', 'gender']]
df_consolidate.head(1)

,platform,customer_id,age,gender
0,android,63ff28e9f22ead6bac895265,0,MALE


In [110]:
df_consolidate.customer_id.nunique()

2270829

In [111]:
df1_merge = pd.merge(df_consolidate,df_metric_data,how='left',on='customer_id')
df2_merge = pd.merge(df1_merge,df_use_case_data,how='left',on='customer_id')
df_classified_data.rename(columns = {'customer_id' :'user_id'}, inplace = True)
df3_merge = pd.merge(df2_merge,df_classified_data,how='left',left_on='customer_id', right_on = 'user_id')

df3_merge = df3_merge[['platform', 'customer_id', 'gender', 'age', 'age_group',
                       'rpc_monthly', 'ltr_segment','life_time_rides', 'life_time_stage', 'service_affinity',
                       'primary_use_case', 'secondary_use_case',  
                       'delivery_food', 'driver_app', 'core_office', 'office', 'core_education', 'gaming',
                       'entertainment', 'social', 'dating', 'finance_investment', 'travel_bookings',  
                       'streaming_music', 'travel_ridehailing', #'rest', 
                       'user_id']]
df3_merge

,platform,customer_id,gender,age,age_group,rpc_monthly,ltr_segment,life_time_rides,life_time_stage,service_affinity,primary_use_case,secondary_use_case,delivery_food,driver_app,core_office,office,core_education,gaming,entertainment,social,dating,finance_investment,travel_bookings,streaming_music,travel_ridehailing,user_id
0,android,63ff28e9f22ead6bac895265,MALE,0,26-35,8,PHH,37.0,HOOK,ONLY_LINK,residential,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,63ff28e9f22ead6bac895265
1,android,5cb7f15554bc7263ff3f4e73,MALE,28,26-35,17,PHH,95.0,COMMITTED,ONLY_AUTO,health_and_personal,place_of_worship,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,5cb7f15554bc7263ff3f4e73
2,android,62428c959d4b75f09cb3a511,FEMALE,0,36-60,30,PHH,248.0,COMMITTED,ONLY_AUTO,household_needs,transit_station,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,62428c959d4b75f09cb3a511
3,android,5fbaaf067e483f4a13fd9b9e,MALE,28,26-35,15,PHH,87.0,COMMITTED,ONLY_AUTO,residential,health_and_personal,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,5fbaaf067e483f4a13fd9b9e
4,android,64d0d9ebc810cc925f36ceea,FEMALE,0,26-35,50,PHH,119.0,COMMITTED,ONLY_AUTO,residential,educational,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,64d0d9ebc810cc925f36ceea
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2287535,android,65e6a82fd8310c2a5f675b0d,MALE,0,26-35,1,HH,1.0,DORMANT,ONLY_LINK,office,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,65e6a82fd8310c2a5f675b0d
2287536,android,65eb2f7ff2a98f9872e52218,MALE,0,21-25,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,65eb2f7ff2a98f9872e52218
2287537,android,62b13e3fc6b74847db76db77,MALE,0,26-35,2,PHH,41.0,HOOK,ONLY_AUTO,residential,NaN,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,62b13e3fc6b74847db76db77
2287538,android,649bb175e318464c7066d22d,MALE,0,26-35,1,HH,1.0,HANDHOLDING,ONLY_AUTO,NaN,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,649bb175e318464c7066d22d


In [112]:
df3_merge.customer_id.nunique()

2270829

### Derived values

In [113]:
df3_merge['age_presence'] = np.where(df3_merge['age'] > 0, 'AVAILABLE', 'NA') 
df3_merge['use_case_presence'] = np.where((df3_merge['primary_use_case'].isna()) 
                                          & 
                                          (df3_merge['secondary_use_case'].isna()), 'NA', 'AVAILABLE')

df3_merge['appo_presence'] = np.where((df3_merge['user_id'].isna() == True),
                                      'NA', 'AVAILABLE')

In [114]:
df3_merge.head(1)

,platform,customer_id,gender,age,age_group,rpc_monthly,ltr_segment,life_time_rides,life_time_stage,service_affinity,primary_use_case,secondary_use_case,delivery_food,driver_app,core_office,office,core_education,gaming,entertainment,social,dating,finance_investment,travel_bookings,streaming_music,travel_ridehailing,user_id,age_presence,use_case_presence,appo_presence
0,android,63ff28e9f22ead6bac895265,MALE,0,26-35,8,PHH,37.0,HOOK,ONLY_LINK,residential,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,63ff28e9f22ead6bac895265,NA,AVAILABLE,AVAILABLE


In [115]:
df3_merge.customer_id.nunique()

2270829

In [116]:
df_temp = df_all_customer[df_all_customer['platform'] == 'android']\
                .groupby(['platform', 'age_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['total'] = df_temp.groupby('platform')['customers'].transform('sum')
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp['total'], 2)
df_temp

,platform,age_presence,customers,total,% distribution
0,android,AVAILABLE,755034,2270829,33.25
1,android,NA,1515795,2270829,66.75


In [117]:
df_temp = df3_merge\
                .groupby(['age_presence','use_case_presence','appo_presence'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()

df_temp['% distribution'] = round(df_temp['customers']*100.00/df3_merge.customer_id.nunique(), 2)
df_temp

,age_presence,use_case_presence,appo_presence,customers,% distribution
0,AVAILABLE,AVAILABLE,AVAILABLE,489061,21.54
1,AVAILABLE,AVAILABLE,NA,85317,3.76
2,AVAILABLE,NA,AVAILABLE,134282,5.91
3,AVAILABLE,NA,NA,46872,2.06
4,NA,AVAILABLE,AVAILABLE,956485,42.12
5,NA,AVAILABLE,NA,168204,7.41
6,NA,NA,AVAILABLE,298023,13.12
7,NA,NA,NA,93655,4.12


In [119]:
df_temp['% distribution'].sum()

100.03999999999999

In [120]:
df3_merge.head()

,platform,customer_id,gender,age,age_group,rpc_monthly,ltr_segment,life_time_rides,life_time_stage,service_affinity,primary_use_case,secondary_use_case,delivery_food,driver_app,core_office,office,core_education,gaming,entertainment,social,dating,finance_investment,travel_bookings,streaming_music,travel_ridehailing,user_id,age_presence,use_case_presence,appo_presence
0,android,63ff28e9f22ead6bac895265,MALE,0,26-35,8,PHH,37.0,HOOK,ONLY_LINK,residential,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,63ff28e9f22ead6bac895265,NA,AVAILABLE,AVAILABLE
1,android,5cb7f15554bc7263ff3f4e73,MALE,28,26-35,17,PHH,95.0,COMMITTED,ONLY_AUTO,health_and_personal,place_of_worship,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,5cb7f15554bc7263ff3f4e73,AVAILABLE,AVAILABLE,AVAILABLE
2,android,62428c959d4b75f09cb3a511,FEMALE,0,36-60,30,PHH,248.0,COMMITTED,ONLY_AUTO,household_needs,transit_station,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,62428c959d4b75f09cb3a511,NA,AVAILABLE,AVAILABLE
3,android,5fbaaf067e483f4a13fd9b9e,MALE,28,26-35,15,PHH,87.0,COMMITTED,ONLY_AUTO,residential,health_and_personal,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,5fbaaf067e483f4a13fd9b9e,AVAILABLE,AVAILABLE,AVAILABLE
4,android,64d0d9ebc810cc925f36ceea,FEMALE,0,26-35,50,PHH,119.0,COMMITTED,ONLY_AUTO,residential,educational,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,64d0d9ebc810cc925f36ceea,NA,AVAILABLE,AVAILABLE


In [121]:
df3_merge.customer_id.nunique()

2270829

### Confidence level

In [122]:
df3_merge.age_group.unique()

array(['26-35', '36-60', '21-25', nan, '15-20'], dtype=object)

In [123]:
def confidence(row):
    
    if row['age_presence'] == 'AVAILABLE' :
        return 'Age Exist'
    

    elif row['appo_presence'] == 'NA' :        
        return 'Unknown' 
    
    
    elif row['use_case_presence'] == 'AVAILABLE' and row['appo_presence'] == 'AVAILABLE' :
        
        
        if row['core_office'] == 1 and row['core_education'] == 1 :
        
            if row['age_group'] in ['15-20', '21-25'] :
                return 'High'
            
            elif row['age_group'] in ['26-35'] :
                return 'Medium'
            
            elif row['age_group'] in ['36-60'] :
                return 'Low'
            
            
            else:
                return 'check'
            
        
        ##Office
        
        elif row['core_office'] == 1 and row['age_group'] in ['26-35', '36-60'] :
            
            if row['primary_use_case'] == 'office' or row['secondary_use_case'] == 'office' :
                return 'High'
        
            elif row['finance_investment'] == 1 :
                return 'High'
            
            elif row['office'] == 1 :
                return 'High'
            
            else:  
                return 'Medium'
            
        elif row['core_office'] == 1 and row['age_group'] in ['15-20', '21-25'] :
            
            if row['age_group'] in ['21-25'] :
                return 'Medium'
        
            else :
                return 'Low'


        ## Education 
        
        elif row['core_education'] == 1 :
            
            if row['primary_use_case'] == 'educational' or row['secondary_use_case'] == 'educational' :
                return 'High'
        
            elif row['dating'] == 1 :
                return 'High'

            elif row['age_group'] in ['15-20', '21-25'] :
                return 'High'
            
            elif row['age_group'] not in ['15-20', '21-25'] : 
                
                if row['primary_use_case'] in ['residential', 'office', 'food'] or row['primary_use_case'] in ['residential', 'office', 'food'] :
                    return 'High'
            
                else :
                    return 'Medium'
            
            else: 
                return 'Low'
                
        
    
        ## Dating 
        
        elif row['dating'] == 1 :
            
            if row['age_group'] in ['15-20', '21-25', '26-35'] :
                return 'High'
            
            elif row['finance_investment'] == 1 :
                return 'High'
        
            else: 
                return 'Low'
            
            
        ## Driver App 
        
        elif row['driver_app'] == 1 :
            
            if row['age_group'] in ['21-25', '26-35'] and row['gender'] == 'MALE' :
                return 'High'
            
            elif row['age_group'] in ['36-60'] :
                return 'Medium'
            
            elif row['gender'] == 'FEMALE' :
                return 'Low'
        
            else: 
                return 'Low'   
            
            
        ## Driver App 
        
        elif row['finance_investment'] == 1 :
            
            if row['age_group'] in ['26-35', '36-60'] and row['office'] == 1 :
                return 'High'
            
            elif row['age_group'] in ['26-35'] and (row['office'] == 1 or row['primary_use_case'] == 'educational' or row['secondary_use_case'] == 'educational'):
                return 'Medium'
            
            elif row['age_group'] in ['21-25'] or row['primary_use_case'] == 'office' or row['secondary_use_case'] == 'office' :
                return 'Low'
        
            else: 
                return 'Medium'   
        
        elif row['gaming'] == 1 :

            if row['age_group'] in ['15-20', '21-25', '26-35'] :
                return 'High'

            else: 
                return 'Low'   


        elif row['delivery_food'] == 1 and row['office'] == 1 and row['entertainment'] == 1 :

            if row['age_group'] in ['26-35'] :
                return 'High'

            else: 
                return 'Medium'        
            
        elif row['gender'] == 'FEMALE' :

            if row['age_group'] in ['21-25', '26-35', '36-60'] and row['service_affinity'] in ['ONLY_AUTO', 'ONLY_CAB', 'AUTO_CAB'] :
                return 'High'
            
            elif row['age_group'] in ['26-35', '36-60'] and row['service_affinity'] not in ['ONLY_LINK'] :
                return 'Low'

            else: 
                return 'Medium'    
            
        elif row['primary_use_case'] == 'office' or row['secondary_use_case'] == 'office' :

            if row['age_group'] in ['26-35', '36-60'] :
                return 'High'

            elif row['age_group'] in ['21-25'] :
                return 'Medium'

            else: 
                return 'Low'    

        elif row['ltr_segment'] == 'HH' :

            if row['primary_use_case'] != None :
                return 'Medium'

            elif row['secondary_use_case'] == None :
                return 'Low'
            
        elif row['delivery_food'] == 1 and row['travel_ridehailing'] == 1 :

            if row['age_group'] in ['26-35'] :
                return 'Medium'

            else :
                return 'Low'
            
        elif row['primary_use_case'] in ['educational'] :

            if row['age_group'] in ['15-20', '21-25'] :
                return 'Medium'

            else :
                return 'Low'

            
        else:
            return 'Unknown'

In [133]:
df3_merge['confidence_tag'] = df3_merge.apply(confidence, axis=1)
df3_merge['confidence_tag'] = df3_merge['confidence_tag'].fillna('Low')

In [134]:
df_temp = df3_merge\
                .groupby(['age_presence','use_case_presence','appo_presence'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()

df_temp['% distribution'] = round(df_temp['customers']*100.00/df3_merge.customer_id.nunique(), 2)
df_temp

,age_presence,use_case_presence,appo_presence,customers,% distribution
0,AVAILABLE,AVAILABLE,AVAILABLE,489061,21.54
1,AVAILABLE,AVAILABLE,NA,85317,3.76
2,AVAILABLE,NA,AVAILABLE,134282,5.91
3,AVAILABLE,NA,NA,46872,2.06
4,NA,AVAILABLE,AVAILABLE,956485,42.12
5,NA,AVAILABLE,NA,168204,7.41
6,NA,NA,AVAILABLE,298023,13.12
7,NA,NA,NA,93655,4.12


In [135]:
df_temp['% distribution'].sum()

100.03999999999999

In [136]:
df_temp = df3_merge\
                .groupby(['age_presence','use_case_presence','appo_presence', 'confidence_tag'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()

df_temp['% distribution'] = round(df_temp['customers']*100.00/df3_merge.customer_id.nunique(), 2)
df_temp

,age_presence,use_case_presence,appo_presence,confidence_tag,customers,% distribution
0,AVAILABLE,AVAILABLE,AVAILABLE,Age Exist,489061,21.54
1,AVAILABLE,AVAILABLE,NA,Age Exist,85317,3.76
2,AVAILABLE,NA,AVAILABLE,Age Exist,134282,5.91
3,AVAILABLE,NA,NA,Age Exist,46872,2.06
4,NA,AVAILABLE,AVAILABLE,High,542965,23.91
5,NA,AVAILABLE,AVAILABLE,Low,91665,4.04
6,NA,AVAILABLE,AVAILABLE,Medium,215107,9.47
7,NA,AVAILABLE,AVAILABLE,Unknown,106776,4.70
8,NA,AVAILABLE,NA,Unknown,168204,7.41
9,NA,NA,AVAILABLE,Low,298023,13.12


In [137]:
df_temp['% distribution'].sum()

100.04

In [138]:
df3_merge[df3_merge['confidence_tag'].isna()]

,platform,customer_id,gender,age,age_group,rpc_monthly,ltr_segment,life_time_rides,life_time_stage,service_affinity,primary_use_case,secondary_use_case,delivery_food,driver_app,core_office,office,core_education,gaming,entertainment,social,dating,finance_investment,travel_bookings,streaming_music,travel_ridehailing,user_id,age_presence,use_case_presence,appo_presence,confidence_tag


In [139]:
df_temp = df3_merge\
                .groupby(['age_presence', 'appo_presence', 'confidence_tag'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()

df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,age_presence,appo_presence,confidence_tag,customers,% distribution
0,AVAILABLE,AVAILABLE,Age Exist,622942,27.43
1,AVAILABLE,NA,Age Exist,132092,5.82
2,NA,AVAILABLE,High,542965,23.91
3,NA,AVAILABLE,Low,389639,17.16
4,NA,AVAILABLE,Medium,215107,9.47
5,NA,AVAILABLE,Unknown,106776,4.70
6,NA,NA,Unknown,261726,11.52


In [140]:
df_temp = df3_merge\
                .groupby(['age_presence', 'confidence_tag'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()

df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,age_presence,confidence_tag,customers,% distribution
0,AVAILABLE,Age Exist,755034,33.24
1,NA,High,542965,23.91
2,NA,Low,389639,17.16
3,NA,Medium,215107,9.47
4,NA,Unknown,368502,16.22


In [141]:
df_temp['% distribution'].sum()

100.0

### Final Dataset

In [142]:
df3_merge[['platform','customer_id','age','age_group']]

,platform,customer_id,age,age_group
0,android,63ff28e9f22ead6bac895265,0,26-35
1,android,5cb7f15554bc7263ff3f4e73,28,26-35
2,android,62428c959d4b75f09cb3a511,0,36-60
3,android,5fbaaf067e483f4a13fd9b9e,28,26-35
4,android,64d0d9ebc810cc925f36ceea,0,26-35
...,...,...,...,...
2287535,android,65e6a82fd8310c2a5f675b0d,0,26-35
2287536,android,65eb2f7ff2a98f9872e52218,0,21-25
2287537,android,62b13e3fc6b74847db76db77,0,26-35
2287538,android,649bb175e318464c7066d22d,0,26-35
